# 10 — Komparasi Head-to-Head Teroptimasi Biner: `PM3-Joint` & `PM3-09 (APF)` vs `MVCBM` & `Barthwal et al. (2026)` ($N = 578$)

Notebook ini menjalankan perbandingan **adil, terkalibrasi biner, bebas kebocoran data pasien, dan sama-sama dioptimalkan pada tugas biner asli** ($N = 578$ pasien, 5-fold stratified CV `seed=1`):
- **Binary Diagnosis:** `Appendicitis (382)` vs `No Appendicitis (196)`
- **Binary Severity (Original Cohort Label):** `Complicated (96)` vs `Uncomplicated + No Appendicitis (482)`

### Kelompok Model yang Dibandingkan & Dioptimalkan Ulang
1. **Model Usulan Teroptimasi Biner (Image-Only & Multimodal `clean_28` + `USConfig`):**
   - **`PM3-Joint Binary`:** Shared `Stage 7` EfficientNet-B0 + dual Cross-Attention MIL heads (`agg_diag`, `agg_sev`) + 8 konsep PM3, dilatih langsung dengan loss biner Diagnosis (balanced) & Severity ($N=578$, `pos_weight=sqrt(n_neg/n_pos)`) tanpa tarikan NLL 3-kelas, plus *checkpoint* terbaik independen per *head*.
   - **`PM3-09 (APF) Binary`:** *Dual-Stage7 Fork* (`stage7_diag` & `stage7_sev`), `CompactCrossAttentionMIL` untuk Diagnosis (Boundary-Focused Focal BCE) + `ClinicalGatedAttentionMIL` untuk Severity (`N=578`), regularisasi asimetris, *Model EMA (0.98)*, *Loss-Guarded monitor*, dan **Asynchronous Partial Freeze**.
2. **Replikasi Paper 1 — `MVCBM` (Marcinkevičs et al., 2024, *MedIA* 91:103042):**
   - `MVCBM-Sequential` (default resmi `config_app_mvcbm.yaml`) & opsi `MVCBM-Joint` dengan **`MVC-Mean` (`avg`)** dan **`MVC-LSTM` (`lstm`)** pada **Image-Only** dan **Multimodal (`fusion=True`, `clean_28`)**, menggunakan bobot lokal `resnet18-5c106cde.pth` dan 9 konsep Tabel 2.
3. **Replikasi Paper 2 — `Barthwal et al. (2026)` (*Egypt Pediatr Assoc Gaz* 74:38) — Uji Bukti Overklaim:**
   - Arsitektur `ResNet-18 + TabNet (n_d=n_a=64, 5 steps, sparsemax) + Cross-Attention + Dense(64->128->64->1)` diuji pada:
     - **Rezim A (`paper_leaky53`):** 53 fitur klinis termasuk variabel bocor (`Length_of_Stay`, `PAS`, `Alvarado_Score`, temuan US) + citra USG mentah *aspect-ratio zero-pad `224x224`* tanpa Telea/CLAHE/mask dan tanpa flip horizontal.
     - **Rezim B (`our_clean28`):** Model Barthwal yang persis sama dilatih ulang pada **28 fitur klinis bersih (`clean_28`)** + citra terproses `USConfig`.


In [1]:
import copy, gc, hashlib, json, math, os, random, sys, time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0, ResNet18_Weights
from scipy.stats import binomtest, rankdata
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

candidates = [Path.cwd(), *Path.cwd().parents]
PM3_DIR = next(
    (p if (p / "pm3_data.py").is_file() else p / "Proposed Model 3")
    for p in candidates
    if (p / "pm3_data.py").is_file() or (p / "Proposed Model 3" / "pm3_data.py").is_file()
)
ROOT = PM3_DIR.parent
if str(PM3_DIR) not in sys.path:
    sys.path.insert(0, str(PM3_DIR))

from pm3_data import (
    BagTransform,
    CONCEPTS_PM3,
    ClinicalPipeline,
    LeakyClinicalPipeline,
    PM3PatientBagDataset,
    USConfig,
    _load_cached_npz,
    build_image_cache,
    collate_patients,
    encode_pm3_concepts,
    images_by_patient,
    load_cohort,
    make_stratified_folds,
)
from pm3_model import ClinicalEncoder, ResidualFiLM, ScaledDotProductCrossAttentionMIL
from pm3_train import save_json, set_seed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM free/total (GiB):",
          [round(x / 1024**3, 2) for x in torch.cuda.mem_get_info()])
else:
    print("Device:", DEVICE)

RUNS_DIR = PM3_DIR / "runs"
OUT_ROOT = RUNS_DIR / "10_paper_replications"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RESNET18_LOCAL_PTH = ROOT / "Code Paper Referensi 1" / "semi-supervised-multiview-cbm" / "pretrained_models" / "resnet" / "resnet18-5c106cde.pth"
print("Local ResNet-18 weights exists:", RESNET18_LOCAL_PTH.is_file())

# ---------------------------------------------------------------------------
# Saklar Eksekusi Eksperimen
# ---------------------------------------------------------------------------
RUN_PM3_BINARY     = True   # PM3-Joint Binary & PM3-09 APF Binary (Image-Only & Multimodal, N=578)
RUN_MVCBM_SEQ      = True   # Resmi Marcinkevics et al. (2024): Sequential (avg & lstm; Image-Only & Multimodal)
RUN_MVCBM_JOINT    = False  # End-to-end Joint MVCBM (opsional lanjutan, SKIP_IF_DONE=True aman disambung)
RUN_BARTHWAL_PAPER = True   # Barthwal et al. (2026) preprocessing persis paper (leaky_53 + 224x224 zero-pad)
RUN_BARTHWAL_CLEAN = True   # Barthwal et al. (2026) preprocessing bersih kita (clean_28 + USConfig)
SKIP_IF_DONE       = True

# ---------------------------------------------------------------------------
# Data Kohort (578 Pasien) & 5-Fold CV Identik (seed=1)
# ---------------------------------------------------------------------------
cohort = load_cohort(ROOT / "app_data_filtered.csv")
manifest = pd.read_csv(ROOT / "image_manifest_filtered.csv")
manifest["image_id"] = manifest["image_id"].astype(str)
manifest = manifest.loc[manifest["US_Number"].isin(cohort.index)].copy().reset_index(drop=True)
folds = make_stratified_folds(cohort, seed=1, n_splits=5)
bags_all = images_by_patient(manifest, bag_mode="all")
image_cache_clean = build_image_cache(manifest, ROOT / "US_Pictures_clean", USConfig(), PM3_DIR / "cache")

# ---------------------------------------------------------------------------
# 9 Konsep Biner Resmi Marcinkevics et al. (2024) Tabel 2 & generate_app_data.py
# ---------------------------------------------------------------------------
CONCEPTS_MVCBM_9 = (
    "Appendix_on_US",
    "Free_Fluids",
    "Appendix_Wall_Layers",
    "Target_Sign",
    "Surrounding_Tissue_Reaction",
    "Pathological_Lymph_Nodes",
    "Bowel_Wall_Thickening",
    "Coprostasis",
    "Meteorism",
)

def encode_mvcbm_concepts_9(df: pd.DataFrame) -> np.ndarray:
    cols = []
    for name in CONCEPTS_MVCBM_9:
        s = df[name].astype("string").str.strip().str.lower()
        if name == "Appendix_Wall_Layers":
            v = s.isin(["raised", "partially raised", "upset"]).fillna(False).to_numpy(dtype=np.float32)
        else:
            v = s.eq("yes").fillna(False).to_numpy(dtype=np.float32)
        cols.append(v)
    return np.stack(cols, axis=1)

# ---------------------------------------------------------------------------
# Preprocessing Citra Persis Barthwal et al. (2026): 224x224 Gray->3ch Zero-Pad
# ---------------------------------------------------------------------------
_BARTHWAL_RAM_CACHE: Dict[Path, np.ndarray] = {}

@dataclass
class BarthwalUSConfig:
    size: int = 224

def preprocess_barthwal_image(path: str | Path, size: int = 224) -> np.ndarray:
    with Image.open(path) as im:
        gray = np.asarray(im.convert("L"))
    h, w = gray.shape
    scale = min(size / w, size / h)
    rw, rh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(gray, (rw, rh), interpolation=interp)
    canvas = np.zeros((size, size), dtype=np.uint8)
    top, left = (size - rh) // 2, (size - rw) // 2
    canvas[top:top + rh, left:left + rw] = resized
    return canvas

def build_barthwal_cache(manifest_df: pd.DataFrame, image_root: Path, cache_dir: Path, size: int = 224) -> Dict[str, Path]:
    dest_dir = cache_dir / f"barthwal_{size}"
    dest_dir.mkdir(parents=True, exist_ok=True)
    cache = {}
    for i, row in enumerate(manifest_df.itertuples(index=False), 1):
        src = image_root / row.path
        st = src.stat()
        key = hashlib.sha256(f"{src.resolve()}:{st.st_size}:{st.st_mtime_ns}:{size}".encode()).hexdigest()[:12]
        dest = dest_dir / f"{row.image_id}_{key}.npz"
        if not dest.exists():
            img = preprocess_barthwal_image(src, size=size)
            np.savez_compressed(dest, image=img)
        cache[str(row.image_id)] = dest
        if i % 500 == 0:
            print(f"Cache Barthwal {size}x{size}: {i}/{len(manifest_df)}", flush=True)
    return cache

def _load_barthwal_npz(path: Path) -> np.ndarray:
    hit = _BARTHWAL_RAM_CACHE.get(path)
    if hit is not None:
        return hit
    with np.load(path) as arr:
        im = arr["image"].copy()
    _BARTHWAL_RAM_CACHE[path] = im
    return im

def compute_fold_pixel_stats(pids: List[int], bags: Dict[int, List[str]], cache: Dict[str, Path]) -> Tuple[float, float]:
    s1, s2, count = 0.0, 0.0, 0
    for pid in pids:
        for iid in bags[int(pid)]:
            x = _load_barthwal_npz(cache[iid]).astype(np.float64) / 255.0
            s1 += float(x.sum())
            s2 += float((x * x).sum())
            count += x.size
    mean = s1 / max(count, 1)
    var = max(s2 / max(count, 1) - mean * mean, 1e-8)
    return float(mean), float(math.sqrt(var))

class BarthwalImageTransform:
    # Sesuai Tabel paper: +-10 deg rotasi, +-5% translasi, +-10% zoom, brightness, Gaussian noise, TANPA flip horizontal
    def __init__(self, train: bool, mean: float, std: float):
        self.train = train
        self.mean = float(mean)
        self.std = float(std)

    def __call__(self, gray_u8: np.ndarray) -> torch.Tensor:
        img = gray_u8
        if self.train:
            h, w = img.shape
            angle = float(np.random.uniform(-10.0, 10.0))
            scale = float(np.random.uniform(0.90, 1.10))
            tx = float(np.random.uniform(-0.05, 0.05)) * w
            ty = float(np.random.uniform(-0.05, 0.05)) * h
            mat = cv2.getRotationMatrix2D((w * 0.5, h * 0.5), angle, scale)
            mat[0, 2] += tx
            mat[1, 2] += ty
            img = cv2.warpAffine(img, mat, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            x = img.astype(np.float32) * (1.0 / 255.0)
            brightness = float(np.random.uniform(-0.10, 0.10))
            x = x + brightness
            if float(np.random.random()) < 0.5:
                x = x + np.random.normal(0.0, 0.02, size=x.shape).astype(np.float32)
            np.clip(x, 0.0, 1.0, out=x)
        else:
            x = img.astype(np.float32) * (1.0 / 255.0)
        x_norm = (x - self.mean) / max(self.std, 1e-6)
        rgb = np.repeat(x_norm[None, :, :], 3, axis=0)
        return torch.from_numpy(np.ascontiguousarray(rgb, dtype=np.float32))

class PaperBagDataset(Dataset):
    def __init__(
        self,
        ids: List[int],
        cohort_df: pd.DataFrame,
        clinical_pipeline,
        bags: Dict[int, List[str]],
        cache: Dict[str, Path],
        task_mode: str,
        image_regime: str = "usconfig",
        train: bool = False,
        pixel_stats: Tuple[float, float] = (0.485, 0.229),
    ):
        assert task_mode in ("binary_diag", "binary_sev")
        assert image_regime in ("usconfig", "barthwal_224")
        self.pids = [int(i) for i in ids]
        self.rows = cohort_df.loc[self.pids]
        self.tabular = clinical_pipeline.transform(self.rows) if clinical_pipeline is not None else np.zeros((len(self.pids), 0), dtype=np.float32)
        raw_lbl = self.rows["label"].to_numpy(int)
        self.y_bin = (raw_lbl > 0).astype(np.float32) if task_mode == "binary_diag" else (raw_lbl == 2).astype(np.float32)
        self.raw_labels = raw_lbl
        self.concepts_9 = encode_mvcbm_concepts_9(self.rows)
        self.bags = bags
        self.cache = cache
        self.image_regime = image_regime
        self.transform = BagTransform(train=train) if image_regime == "usconfig" else BarthwalImageTransform(train=train, mean=pixel_stats[0], std=pixel_stats[1])

    def __len__(self) -> int:
        return len(self.pids)

    def __getitem__(self, i: int) -> dict:
        pid = self.pids[i]
        iids = self.bags[pid]
        if self.image_regime == "usconfig":
            imgs = [self.transform(*_load_cached_npz(self.cache[iid]))[0] for iid in iids]
        else:
            imgs = [self.transform(_load_barthwal_npz(self.cache[iid])) for iid in iids]
        return {
            "patient_id": pid,
            "tabular": torch.from_numpy(self.tabular[i]),
            "y_bin": float(self.y_bin[i]),
            "raw_label": int(self.raw_labels[i]),
            "concepts_9": torch.from_numpy(self.concepts_9[i]),
            "images": torch.stack(imgs),
            "image_ids": iids,
        }

class LockedBNResNet18(nn.Module):
    def __init__(self, local_pth: Path = RESNET18_LOCAL_PTH):
        super().__init__()
        if local_pth.is_file():
            net = models.resnet18(weights=None)
            state = torch.load(local_pth, map_location="cpu", weights_only=False)
            net.load_state_dict(state)
        else:
            net = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        net.fc = nn.Identity()
        self.net = net

    def train(self, mode: bool = True):
        super().train(mode)
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

def load_resnet18_backbone(local_pth: Path = RESNET18_LOCAL_PTH) -> nn.Module:
    return LockedBNResNet18(local_pth=local_pth)

def encode_resnet18_microbatched(backbone: nn.Module, all_imgs: torch.Tensor, microbatch: int = 16) -> torch.Tensor:
    n = all_imgs.size(0)
    rem = n % microbatch
    if rem != 0:
        pad = torch.zeros((microbatch - rem, *all_imgs.shape[1:]), dtype=all_imgs.dtype, device=all_imgs.device)
        inp = torch.cat([all_imgs, pad], dim=0)
    else:
        inp = all_imgs
    outs = [backbone(inp[i:i + microbatch]) for i in range(0, inp.size(0), microbatch)]
    return torch.cat(outs, dim=0)[:n]

def compute_ece(y_true: np.ndarray, probs: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for b0, b1 in zip(bins[:-1], bins[1:]):
        mask = (probs >= b0) & (probs <= b1 if b1 == 1.0 else probs < b1)
        if mask.any():
            ece += (mask.sum() / n) * abs(float(y_true[mask].mean()) - float(probs[mask].mean()))
    return float(ece)

def compute_binary_metrics(y_true: np.ndarray, probs: np.ndarray, pred: np.ndarray) -> dict:
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(probs, dtype=float)
    pr = np.asarray(pred, dtype=int)
    cm = confusion_matrix(y, pr, labels=[0, 1])
    tn, fp, fn, tp = (int(v) for v in cm.ravel())
    auroc = float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float("nan")
    aupr = float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else float("nan")
    return {
        "n": int(len(y)),
        "n_pos": int((y == 1).sum()),
        "accuracy": float(accuracy_score(y, pr)),
        "precision": float(precision_score(y, pr, zero_division=0)),
        "sensitivity": float(tp / max(tp + fn, 1)),
        "specificity": float(tn / max(tn + fp, 1)),
        "f1": float(f1_score(y, pr, zero_division=0)),
        "macro_f1": float(f1_score(y, pr, average="macro", zero_division=0)),
        "auroc": auroc,
        "aupr": aupr,
        "brier": float(brier_score_loss(y, p)),
        "ece": compute_ece(y, p),
        "confusion_matrix": cm.tolist(),
    }

def calibrate_binary_oof(oof: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    df = oof.copy()
    df["tau_used"] = 0.50
    df["pred_calibrated"] = (df["prob"].to_numpy(float) >= 0.50).astype(int)
    t_grid = np.round(np.arange(0.30, 0.71, 0.02), 2)
    for fk in sorted(df["fold"].unique()):
        tr = df["fold"] != fk
        va = df["fold"] == fk
        p_tr, y_tr = df.loc[tr, "prob"].to_numpy(float), df.loc[tr, "label"].to_numpy(int)
        best_score, best_t = (-1.0, -999.0), 0.50
        for t in t_grid:
            pr = (p_tr >= t).astype(int)
            f1m = float(f1_score(y_tr, pr, labels=[0, 1], average="macro", zero_division=0))
            cand = (f1m, -abs(float(t) - 0.50))
            if cand > best_score:
                best_score, best_t = cand, float(t)
        df.loc[va, "tau_used"] = best_t
        df.loc[va, "pred_calibrated"] = (df.loc[va, "prob"].to_numpy(float) >= best_t).astype(int)
    cal = compute_binary_metrics(df["label"].to_numpy(int), df["prob"].to_numpy(float), df["pred_calibrated"].to_numpy(int))
    return df, cal

def _fast_auroc(y: np.ndarray, s: np.ndarray) -> float:
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = rankdata(s, method="average")
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n0 * n1))

def paired_bootstrap_delta_auroc(y: np.ndarray, p_a: np.ndarray, p_b: np.ndarray, n_boot: int = 3000, seed: int = 1) -> Tuple[float, float, float, float]:
    rng = np.random.default_rng(seed)
    idx0, idx1 = np.flatnonzero(y == 0), np.flatnonzero(y == 1)
    n0, n1 = len(idx0), len(idx1)
    obs = _fast_auroc(y, p_a) - _fast_auroc(y, p_b)
    y_boot = np.concatenate([np.zeros(n0, dtype=int), np.ones(n1, dtype=int)])
    deltas = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        b_idx = np.concatenate([rng.choice(idx0, n0, replace=True), rng.choice(idx1, n1, replace=True)])
        deltas[i] = _fast_auroc(y_boot, p_a[b_idx]) - _fast_auroc(y_boot, p_b[b_idx])
    p_val = float(2.0 * min((deltas <= 0).mean(), (deltas >= 0).mean()))
    return float(obs), float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5)), min(p_val, 1.0)

def exact_mcnemar_p(y: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray) -> float:
    ok_a = (pred_a == y)
    ok_b = (pred_b == y)
    b = int((ok_a & (~ok_b)).sum())
    c = int(((~ok_a) & ok_b).sum())
    return float(binomtest(b, b + c, p=0.5).pvalue) if (b + c) > 0 else 1.0

def save_binary_run_artifacts(
    out_dir: Path,
    variant_name: str,
    oof_df: pd.DataFrame,
    fold_rows: List[dict],
    extra_meta: Optional[dict] = None,
) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    oof_df = oof_df.sort_values("patient_id").reset_index(drop=True)
    oof_cal_df, cal_metrics = calibrate_binary_oof(oof_df)
    fix_metrics = compute_binary_metrics(
        oof_cal_df["label"].to_numpy(int),
        oof_cal_df["prob"].to_numpy(float),
        (oof_cal_df["prob"].to_numpy(float) >= 0.50).astype(int),
    )
    oof_cal_df.to_csv(out_dir / "oof_predictions.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(out_dir / "fold_metrics.csv", index=False)
    save_json(out_dir / "oof_metrics_fixed.json", fix_metrics)
    save_json(out_dir / "oof_metrics_calibrated.json", cal_metrics)
    if extra_meta:
        save_json(out_dir / "config.json", extra_meta)
    print(
        f"[SAVED] {variant_name:36s} | AUROC={cal_metrics['auroc']:.4f} | AUPR={cal_metrics['aupr']:.4f} | "
        f"Acc={cal_metrics['accuracy']:.4f} | F1={cal_metrics['f1']:.4f} | Sens={cal_metrics['sensitivity']:.4f} | Spec={cal_metrics['specificity']:.4f}",
        flush=True,
    )
    return {"oof": oof_cal_df, "calibrated": cal_metrics, "fixed": fix_metrics, "folds": pd.DataFrame(fold_rows)}

def _load_saved_binary_run(out_dir: Path) -> Optional[dict]:
    oof_p = out_dir / "oof_predictions.csv"
    cal_p = out_dir / "oof_metrics_calibrated.json"
    fix_p = out_dir / "oof_metrics_fixed.json"
    if oof_p.is_file() and cal_p.is_file() and fix_p.is_file():
        return {
            "oof": pd.read_csv(oof_p),
            "calibrated": json.loads(cal_p.read_text(encoding="utf-8")),
            "fixed": json.loads(fix_p.read_text(encoding="utf-8")),
        }
    return None

print(f"Cohort loaded: N={len(cohort)} | Diagnosis pos={(cohort.label>0).sum()} | Severity pos={(cohort.label==2).sum()}")
print(f"Clean image cache ready: {len(image_cache_clean)} images | Folds={len(folds)}")


GPU: NVIDIA GeForce RTX 3060 Laptop GPU | VRAM free/total (GiB): [5.0, 6.0]
Local ResNet-18 weights exists: True
Preprocessing/cache PM3: 250/1741
Preprocessing/cache PM3: 500/1741
Preprocessing/cache PM3: 750/1741
Preprocessing/cache PM3: 1000/1741
Preprocessing/cache PM3: 1250/1741
Preprocessing/cache PM3: 1500/1741
Cohort loaded: N=578 | Diagnosis pos=382 | Severity pos=96
Clean image cache ready: 1741 images | Folds=5


## 1. Model Usulan Teroptimasi Biner ($N = 578$): `PM3-Joint Binary` & `PM3-09 (Dual-Stage7 APF) Binary`

Berbeda dari NB 02/09 yang mengevaluasi proyeksi biner dari model 3-kelas (sehingga *head* dipengaruhi tarikan NLL 3-kelas), di sini **`PM3-Joint`** dan **`PM3-09 (APF)`** disesuaikan secara langsung agar **optimal pada kedua tugas biner asli (`Diagnosis`: App vs No-App, dan `Severity`: Complicated vs All, $N = 578$)**, baik untuk **Image-Only** maupun **Multimodal**:
- **`PM3-Joint Binary`:** Shared `Stage 7` EfficientNet-B0 + dua cabang Cross-Attention MIL (`agg_diag`, `agg_sev`) + 8 konsep PM3 + (*opsional* `ClinicalEncoder` & `ResidualFiLM`). Kehilangan 3-kelas (`hier_nll`) dihapus dan diganti dengan loss biner seimbang + *independent best checkpoint* per *head*.
- **`PM3-09 (APF) Binary`:** Dual `Stage 7` Fork (`stage7_diag` + `CompactCrossAttentionMIL` + Boundary-Focused Focal BCE; dan `stage7_sev` + `ClinicalGatedAttentionMIL` pada seluruh $N=578$ pasien dengan `pos_weight=sqrt(482/96)` + `comp_entropy_reg`), regularisasi & hiperparameter asimetris, *Model EMA (0.98)*, *Loss-Guarded monitor*, dan **Asynchronous Partial Freeze**.


In [2]:
# ==============================================================================
# Cell 2: PM3-Joint Binary & PM3-09 (APF) Binary — Image-Only & Multimodal
# ==============================================================================
DIAG_CONCEPT_IDX = [0, 1, 2, 3, 5]
COMP_CONCEPT_IDX = [1, 4, 5, 6, 7]

class PM3JointBinaryNet(nn.Module):
    # PM3-Joint (Stage 7 Shared) yang dioptimalkan langsung untuk Binary Diag & Binary Sev (N=578)
    def __init__(self, modality: str = "multimodal", attn_dim: int = 128, dropout: float = 0.35):
        super().__init__()
        assert modality in ("image_only", "multimodal")
        self.modality = modality
        net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.frozen = net.features[:7]
        self.frozen.requires_grad_(False)
        self.frozen.eval()
        self.stage7 = net.features[7]  # 320 channels
        self._stage7_frozen = False

        self.img_proj = nn.Sequential(
            nn.Linear(320, attn_dim), nn.LayerNorm(attn_dim), nn.GELU(), nn.Dropout(dropout)
        )
        clin_dim = 32
        use_cq = (modality == "multimodal")
        self.agg_diag = ScaledDotProductCrossAttentionMIL(attn_dim, 4, 0.7, 0.10, clin_dim, use_cq)
        self.agg_sev = ScaledDotProductCrossAttentionMIL(attn_dim, 4, 0.7, 0.10, clin_dim, use_cq)
        self.concept_head = nn.Linear(attn_dim, len(CONCEPTS_PM3))
        self.aux_img_diag = nn.Linear(attn_dim, 1)
        self.aux_img_sev = nn.Linear(attn_dim, 1)

        if modality == "multimodal":
            self.clin_enc = ClinicalEncoder(28, clin_dim, dropout=0.25)
            self.inst_film = ResidualFiLM(clin_dim, attn_dim)
            self.bag_film_diag = ResidualFiLM(clin_dim, attn_dim)
            self.bag_film_sev = ResidualFiLM(clin_dim, attn_dim)
            self.vis_anchor_diag = nn.Sequential(nn.Linear(attn_dim, 32), nn.LayerNorm(32), nn.GELU())
            self.vis_anchor_sev = nn.Sequential(nn.Linear(attn_dim, 32), nn.LayerNorm(32), nn.GELU())
            self.head_diag = nn.Linear(clin_dim + 32, 1)
            self.head_sev = nn.Linear(clin_dim + 32, 1)
            self.clin_skip_diag = nn.Linear(28, 1)
            self.clin_skip_sev = nn.Linear(28, 1)
            self.mod_drop = 0.15
        else:
            self.head_diag = nn.Linear(attn_dim, 1)
            self.head_sev = nn.Linear(attn_dim, 1)

    def train(self, mode: bool = True):
        super().train(mode)
        self.frozen.eval()
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def set_stage7_frozen(self, frozen: bool):
        self._stage7_frozen = bool(frozen)
        self.stage7.requires_grad_(not self._stage7_frozen)

    def encode_chunked(self, images: torch.Tensor, pixel_masks: torch.Tensor, step: int = 16) -> torch.Tensor:
        n_orig = len(images)
        if n_orig % step != 0:
            pad_n = step - (n_orig % step)
            images = torch.cat((images, images.new_zeros((pad_n, *images.shape[1:]))), dim=0)
            pixel_masks = torch.cat((pixel_masks, pixel_masks.new_ones((pad_n, *pixel_masks.shape[1:]))), dim=0)
        parts = []
        for s in range(0, len(images), step):
            im_c, mk_c = images[s:s + step], pixel_masks[s:s + step]
            with torch.no_grad():
                f6 = self.frozen(im_c)
            if self._stage7_frozen:
                with torch.no_grad():
                    x7 = self.stage7(f6)
            else:
                x7 = self.stage7(f6)
            x7 = x7.float()
            m = F.adaptive_avg_pool2d(mk_c.float(), x7.shape[-2:])
            pooled = (x7 * m).sum((2, 3)) / m.sum((2, 3)).clamp_min(1e-6)
            parts.append(self.img_proj(pooled))
        return torch.cat(parts, dim=0)[:n_orig]

    def forward_bag(self, h_raw: torch.Tensor, tabular: Optional[torch.Tensor] = None) -> dict:
        zd_raw, _ = self.agg_diag(h_raw, None)
        zs_raw, _ = self.agg_sev(h_raw, None)
        concept_logits = self.concept_head(0.5 * (zd_raw + zs_raw)).float()
        aux_d = self.aux_img_diag(zd_raw).float().squeeze()
        aux_s = self.aux_img_sev(zs_raw).float().squeeze()

        if self.modality == "multimodal":
            tab_f = tabular.float()
            e_clin = self.clin_enc(tab_f)
            tab_skip = tab_f
            if self.training and self.mod_drop > 0:
                keep = (torch.rand((), device=e_clin.device) >= self.mod_drop).float()
                scale = 1.0 / (1.0 - self.mod_drop)
                e_clin = e_clin * keep * scale
                tab_skip = tab_skip * keep * scale
            h_mod, _, _ = self.inst_film(h_raw, e_clin)
            zd_vis, _ = self.agg_diag(h_mod, e_clin)
            zs_vis, _ = self.agg_sev(h_mod, e_clin)
            zd_mod, _, _ = self.bag_film_diag(zd_vis, e_clin)
            zs_mod, _, _ = self.bag_film_sev(zs_vis, e_clin)
            logit_d = self.head_diag(torch.cat([e_clin, self.vis_anchor_diag(zd_mod)], dim=-1)).float().squeeze() + self.clin_skip_diag(tab_skip).float().squeeze()
            logit_s = self.head_sev(torch.cat([e_clin, self.vis_anchor_sev(zs_mod)], dim=-1)).float().squeeze() + self.clin_skip_sev(tab_skip).float().squeeze()
        else:
            logit_d = self.head_diag(zd_raw).float().squeeze()
            logit_s = self.head_sev(zs_raw).float().squeeze()
        return {"logit_diag": logit_d, "logit_sev": logit_s, "aux_diag": aux_d, "aux_sev": aux_s, "concept_logits": concept_logits}

    def parameter_groups(self) -> List[dict]:
        bb = list(self.stage7.parameters())
        bb_ids = {id(p) for p in bb}
        attn_mods = [self.img_proj, self.agg_diag, self.agg_sev, self.concept_head]
        at = [p for m in attn_mods for p in m.parameters() if id(p) not in bb_ids]
        used = bb_ids | {id(p) for p in at}
        hd = [p for p in self.parameters() if p.requires_grad and id(p) not in used]
        return [
            {"name": "stage7", "params": bb, "lr": 2.0e-5, "weight_decay": 1e-4},
            {"name": "attn", "params": at, "lr": 2.0e-4, "weight_decay": 1e-3},
            {"name": "head", "params": hd, "lr": 4.0e-4, "weight_decay": 1e-3},
        ]

# --- Komponen PM3-09 (APF) Binary (mendukung Image-Only & Multimodal) ---
@dataclass
class APFBinConfig:
    epochs: int = 26
    warmup_freeze_epochs: int = 3
    scheduler_t_max: int = 22
    scheduler_eta_min: float = 1e-6
    patience_diag: int = 8
    patience_sev: int = 7
    min_freeze_epoch_diag: int = 10
    min_freeze_epoch_sev: int = 9
    batch_size: int = 16
    seed: int = 1
    amp_fp16: bool = True
    bag_view_drop_prob: float = 0.25
    ema_decay: float = 0.98
    ema_start_epoch: int = 3
    loss_guard_penalty: float = 0.12

    diag_dim: int = 96
    diag_clin_hidden: int = 48
    diag_clin_dim: int = 32
    diag_vis_proj_dim: int = 32
    diag_drop2d: float = 0.10
    diag_img_drop: float = 0.35
    diag_clin_drop: float = 0.25
    diag_film_drop: float = 0.20
    diag_head_drop: float = 0.20
    diag_attn_drop: float = 0.10
    diag_attn_temp: float = 0.65
    diag_mod_drop: float = 0.15
    diag_tab_jitter: float = 0.04
    diag_vis_jitter: float = 0.03
    diag_smoothing: float = 0.03
    diag_focal_gamma: float = 1.0
    diag_lr_bb: float = 1.5e-5
    diag_lr_attn: float = 2.0e-4
    diag_lr_head: float = 3.5e-4
    diag_wd_bb: float = 8e-3
    diag_wd_attn: float = 5e-3
    diag_wd_clin: float = 1.5e-2
    diag_wd_head: float = 5e-3
    w_aux_diag: float = 0.45
    w_concept_diag: float = 0.35

    sev_dim: int = 64
    sev_clin_hidden: int = 32
    sev_clin_dim: int = 24
    sev_vis_proj_dim: int = 24
    sev_drop2d: float = 0.20
    sev_img_drop: float = 0.42
    sev_clin_drop: float = 0.35
    sev_film_drop: float = 0.28
    sev_head_drop: float = 0.28
    sev_attn_drop: float = 0.15
    sev_attn_temp: float = 0.45
    sev_mod_drop: float = 0.30
    sev_tab_jitter: float = 0.06
    sev_vis_jitter: float = 0.04
    sev_smoothing: float = 0.04
    sev_entropy_reg: float = 0.015
    sev_lr_bb: float = 1.0e-5
    sev_lr_attn: float = 1.5e-4
    sev_lr_head: float = 2.2e-4
    sev_wd_bb: float = 2.0e-2
    sev_wd_attn: float = 1.2e-2
    sev_wd_clin: float = 2.5e-2
    sev_wd_head: float = 1.5e-2
    w_aux_sev: float = 0.55
    w_concept_sev: float = 0.35

class RegularizedResidualFiLM(nn.Module):
    def __init__(self, cond_dim: int, feat_dim: int, hidden: int = 32, dropout: float = 0.25):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cond_dim, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 2 * feat_dim))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, feat: torch.Tensor, cond: torch.Tensor):
        dg, beta = self.net(cond).chunk(2, dim=-1)
        if feat.dim() == 2 and dg.dim() == 1:
            dg_b, beta_b = dg.unsqueeze(0), beta.unsqueeze(0)
        else:
            dg_b, beta_b = dg, beta
        return (1.0 + dg_b) * feat + beta_b

class CompactClinicalEncoder(nn.Module):
    def __init__(self, in_dim: int = 28, hidden: int = 48, out_dim: int = 32, dropout: float = 0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim), nn.LayerNorm(out_dim), nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x.float())

class CompactCrossAttentionMIL(nn.Module):
    def __init__(self, dim: int = 96, num_heads: int = 4, temperature: float = 0.65,
                 dropout: float = 0.10, clinical_dim: Optional[int] = 32):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = math.sqrt(self.head_dim) * temperature
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim)
        self.q_base = nn.Parameter(torch.randn(dim) * 0.03)
        self.q_clin = nn.Linear(clinical_dim, dim) if clinical_dim is not None else None
        if self.q_clin is not None:
            nn.init.zeros_(self.q_clin.weight)
            nn.init.zeros_(self.q_clin.bias)

    def forward(self, h: torch.Tensor, e_clin: Optional[torch.Tensor] = None):
        k_len = h.shape[0]
        q_vec = self.q_base + (self.q_clin(e_clin) if (e_clin is not None and self.q_clin is not None) else 0.0)
        q = self.q_proj(q_vec).view(self.num_heads, 1, self.head_dim)
        k = self.k_proj(h).view(k_len, self.num_heads, self.head_dim).transpose(0, 1)
        v = self.v_proj(h).view(k_len, self.num_heads, self.head_dim).transpose(0, 1)
        scores = torch.matmul(q.float(), k.float().transpose(-2, -1)).squeeze(1) / self.scale
        w_heads = F.softmax(scores, dim=-1)
        ctx = torch.matmul(self.attn_drop(w_heads).unsqueeze(1), v.float()).squeeze(1).reshape(self.dim)
        w_mean = w_heads.mean(dim=0)
        res = (w_mean[:, None] * h.float()).sum(dim=0)
        return self.norm(self.out_proj(ctx) + res), w_mean

class ClinicalGatedAttentionMIL(nn.Module):
    def __init__(self, dim: int = 64, hidden: int = 32, clinical_dim: Optional[int] = 24,
                 temperature: float = 0.45, dropout: float = 0.15):
        super().__init__()
        self.temperature = temperature
        self.h_norm = nn.LayerNorm(dim)
        self.V = nn.Linear(dim, hidden, bias=True)
        self.U = nn.Linear(dim, hidden, bias=True)
        self.W_c = nn.Linear(clinical_dim, hidden, bias=False) if clinical_dim is not None else None
        self.w = nn.Linear(hidden, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.out_norm = nn.LayerNorm(dim)
        for layer in [l for l in (self.V, self.U, self.W_c, self.w) if l is not None]:
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)

    def forward(self, h: torch.Tensor, e_clin: Optional[torch.Tensor] = None):
        hn = self.h_norm(h.float())
        clin_term = self.W_c(e_clin.float()).unsqueeze(0) if (e_clin is not None and self.W_c is not None) else 0.0
        u = torch.tanh(self.V(hn) + clin_term)
        g = torch.sigmoid(self.U(hn))
        scores = self.w(self.drop(u * g)).squeeze(-1) / self.temperature
        weights = F.softmax(scores, dim=0)
        z = (weights[:, None] * h.float()).sum(dim=0)
        return self.out_norm(z), weights

class APFBinBranch(nn.Module):
    def __init__(self, branch_type: str, modality: str, stage7_module: nn.Module, cfg: APFBinConfig):
        super().__init__()
        assert branch_type in ("diag", "sev") and modality in ("image_only", "multimodal")
        self.branch_type = branch_type
        self.modality = modality
        self.stage7 = stage7_module
        self._stage7_frozen = False
        self._branch_frozen = False

        if branch_type == "diag":
            d, c_h, c_d, v_d = cfg.diag_dim, cfg.diag_clin_hidden, cfg.diag_clin_dim, cfg.diag_vis_proj_dim
            self.drop2d = nn.Dropout2d(cfg.diag_drop2d)
            self.img_proj = nn.Sequential(nn.Linear(320, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(cfg.diag_img_drop))
            self.agg = CompactCrossAttentionMIL(d, 4, cfg.diag_attn_temp, cfg.diag_attn_drop, c_d if modality == "multimodal" else None)
            self.concept_head = nn.Linear(d, len(DIAG_CONCEPT_IDX))
            self.aux_img_head = nn.Linear(d, 1)
            self.vis_anchor = nn.Sequential(nn.Linear(d, v_d), nn.LayerNorm(v_d), nn.GELU())
            if modality == "multimodal":
                self.clin_mlp = CompactClinicalEncoder(28, c_h, c_d, cfg.diag_clin_drop)
                self.clin_skip = nn.Linear(28, 1)
                self.inst_film = RegularizedResidualFiLM(c_d, d, hidden=48, dropout=cfg.diag_film_drop)
                self.bag_film = RegularizedResidualFiLM(c_d, d, hidden=48, dropout=cfg.diag_film_drop)
                self.head = nn.Sequential(nn.Dropout(cfg.diag_head_drop), nn.Linear(c_d + v_d, 1))
            else:
                self.head = nn.Sequential(nn.Dropout(cfg.diag_head_drop), nn.Linear(v_d, 1))
            self.mod_drop, self.tab_jitter, self.vis_jitter = cfg.diag_mod_drop, cfg.diag_tab_jitter, cfg.diag_vis_jitter
        else:
            d, c_h, c_d, v_d = cfg.sev_dim, cfg.sev_clin_hidden, cfg.sev_clin_dim, cfg.sev_vis_proj_dim
            self.drop2d = nn.Dropout2d(cfg.sev_drop2d)
            self.img_proj = nn.Sequential(nn.Linear(320, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(cfg.sev_img_drop))
            self.agg = ClinicalGatedAttentionMIL(d, 32, c_d if modality == "multimodal" else None, cfg.sev_attn_temp, cfg.sev_attn_drop)
            self.concept_head = nn.Linear(d, len(COMP_CONCEPT_IDX))
            self.aux_img_head = nn.Linear(d, 1)
            self.vis_anchor = nn.Sequential(nn.Linear(d, v_d), nn.LayerNorm(v_d), nn.GELU())
            if modality == "multimodal":
                self.clin_mlp = CompactClinicalEncoder(28, c_h, c_d, cfg.sev_clin_drop)
                self.clin_skip = nn.Linear(28, 1)
                self.inst_film = RegularizedResidualFiLM(c_d, d, hidden=32, dropout=cfg.sev_film_drop)
                self.bag_film = RegularizedResidualFiLM(c_d, d, hidden=32, dropout=cfg.sev_film_drop)
                self.head = nn.Sequential(nn.Dropout(cfg.sev_head_drop), nn.Linear(c_d + v_d, 1))
            else:
                self.head = nn.Sequential(nn.Dropout(cfg.sev_head_drop), nn.Linear(v_d, 1))
            self.mod_drop, self.tab_jitter, self.vis_jitter = cfg.sev_mod_drop, cfg.sev_tab_jitter, cfg.sev_vis_jitter

    def set_stage7_frozen(self, frozen: bool):
        self._stage7_frozen = bool(frozen)
        self.stage7.requires_grad_(not (self._stage7_frozen or self._branch_frozen))

    def freeze_entire_branch(self):
        self._branch_frozen = True
        self._stage7_frozen = True
        self.requires_grad_(False)
        self.eval()

    def encode_from_f6(self, f6: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        if self._stage7_frozen or self._branch_frozen:
            with torch.no_grad():
                x = self.stage7(f6)
        else:
            x = self.stage7(f6)
        x = x.float()
        if self.training and not self._branch_frozen:
            x = self.drop2d(x)
        m = F.adaptive_avg_pool2d(masks.float(), x.shape[-2:])
        pooled = (x * m).sum((2, 3)) / m.sum((2, 3)).clamp_min(1e-6)
        if self._branch_frozen:
            with torch.no_grad():
                return self.img_proj(pooled)
        return self.img_proj(pooled)

    def forward_bag(self, h_raw: torch.Tensor, tabular: Optional[torch.Tensor] = None) -> dict:
        ctx = torch.no_grad() if self._branch_frozen else torch.enable_grad()
        with ctx:
            if self.training and not self._branch_frozen and self.vis_jitter > 0:
                h_raw = h_raw + torch.randn_like(h_raw) * self.vis_jitter
            z_raw, attn_raw = self.agg(h_raw, None)
            concept_logits = self.concept_head(z_raw).float()
            aux_img_logit = self.aux_img_head(z_raw).float().squeeze()
            if self.modality == "multimodal":
                tab_f = tabular.float()
                if self.training and not self._branch_frozen and self.tab_jitter > 0:
                    tab_f = tab_f + torch.randn_like(tab_f) * self.tab_jitter
                e_clin = self.clin_mlp(tab_f)
                tab_skip = tab_f
                if self.training and not self._branch_frozen and self.mod_drop > 0:
                    keep = (torch.rand((), device=e_clin.device) >= self.mod_drop).float()
                    scale = 1.0 / (1.0 - self.mod_drop)
                    e_clin = e_clin * keep * scale
                    tab_skip = tab_skip * keep * scale
                h_main = self.inst_film(h_raw, e_clin)
                z_vis, attn_main = self.agg(h_main, e_clin)
                z_mod = self.bag_film(z_vis, e_clin)
                logit = self.head(torch.cat([e_clin, self.vis_anchor(z_mod)], dim=-1)).float().squeeze() + self.clin_skip(tab_skip).float().squeeze()
            else:
                attn_main = attn_raw
                logit = self.head(self.vis_anchor(z_raw)).float().squeeze()
            return {"logit": logit, "prob": logit.sigmoid(), "aux_img_logit": aux_img_logit, "concept_logits": concept_logits, "attention": attn_main}

    def parameter_groups(self, cfg: APFBinConfig) -> List[dict]:
        if self.branch_type == "diag":
            lr_bb, lr_at, lr_hd = cfg.diag_lr_bb, cfg.diag_lr_attn, cfg.diag_lr_head
            wd_bb, wd_at, wd_cl, wd_hd = cfg.diag_wd_bb, cfg.diag_wd_attn, cfg.diag_wd_clin, cfg.diag_wd_head
        else:
            lr_bb, lr_at, lr_hd = cfg.sev_lr_bb, cfg.sev_lr_attn, cfg.sev_lr_head
            wd_bb, wd_at, wd_cl, wd_hd = cfg.sev_wd_bb, cfg.sev_wd_attn, cfg.sev_wd_clin, cfg.sev_wd_head
        bb = list(self.stage7.parameters())
        bb_ids = {id(p) for p in bb}
        at = [p for m in (self.img_proj, self.agg, self.concept_head) for p in m.parameters() if id(p) not in bb_ids]
        at_ids = {id(p) for p in at}
        cl = [p for m in (self.clin_mlp, self.clin_skip) for p in m.parameters() if id(p) not in (bb_ids | at_ids)] if self.modality == "multimodal" else []
        used = bb_ids | at_ids | {id(p) for p in cl}
        hd = [p for p in self.parameters() if id(p) not in used]
        groups = [
            {"name": "stage7", "params": bb, "lr": lr_bb, "weight_decay": wd_bb},
            {"name": "attn_concept", "params": at, "lr": lr_at, "weight_decay": wd_at},
            {"name": "film_head", "params": hd, "lr": lr_hd, "weight_decay": wd_hd},
        ]
        if cl:
            groups.append({"name": "clinical", "params": cl, "lr": lr_hd, "weight_decay": wd_cl})
        return groups

class PM3APF09BinaryNet(nn.Module):
    def __init__(self, modality: str, cfg: APFBinConfig):
        super().__init__()
        self.cfg = cfg
        self.modality = modality
        net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.frozen = net.features[:7]
        self.frozen.requires_grad_(False)
        self.frozen.eval()
        self.branch_diag = APFBinBranch("diag", modality, copy.deepcopy(net.features[7]), cfg)
        self.branch_sev = APFBinBranch("sev", modality, copy.deepcopy(net.features[7]), cfg)

    def train(self, mode: bool = True):
        super().train(mode)
        self.frozen.eval()
        if self.branch_diag._branch_frozen:
            self.branch_diag.eval()
        if self.branch_sev._branch_frozen:
            self.branch_sev.eval()
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def set_stage7_frozen(self, frozen: bool):
        self.branch_diag.set_stage7_frozen(frozen)
        self.branch_sev.set_stage7_frozen(frozen)

    def encode_both_chunked(self, images: torch.Tensor, pixel_masks: torch.Tensor, step: int = 16):
        n_orig = len(images)
        if n_orig % step != 0:
            pad_n = step - (n_orig % step)
            images = torch.cat((images, images.new_zeros((pad_n, *images.shape[1:]))), dim=0)
            pixel_masks = torch.cat((pixel_masks, pixel_masks.new_ones((pad_n, *pixel_masks.shape[1:]))), dim=0)
        hd_parts, hs_parts = [], []
        for s in range(0, len(images), step):
            with torch.no_grad():
                f6 = self.frozen(images[s:s + step])
            hd_parts.append(self.branch_diag.encode_from_f6(f6, pixel_masks[s:s + step]))
            hs_parts.append(self.branch_sev.encode_from_f6(f6, pixel_masks[s:s + step]))
        return torch.cat(hd_parts, dim=0)[:n_orig], torch.cat(hs_parts, dim=0)[:n_orig]

class BranchEMA:
    def __init__(self, branch: nn.Module, decay: float = 0.98):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in branch.state_dict().items()}

    @torch.no_grad()
    def update(self, branch: nn.Module):
        for k, v in branch.state_dict().items():
            if v.is_floating_point():
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                self.shadow[k].copy_(v.detach())

    def apply_to(self, branch: nn.Module) -> dict:
        cur = branch.state_dict()
        bk = {k: cur[k].detach().clone() for k in self.shadow}
        branch.load_state_dict({**cur, **self.shadow}, strict=True)
        return bk

    def restore(self, branch: nn.Module, bk: dict):
        branch.load_state_dict({**branch.state_dict(), **bk}, strict=True)

def compute_pm3_bin_weights(train_df: pd.DataFrame) -> dict:
    labels = train_df["label"].to_numpy(int)
    counts = np.bincount(labels, minlength=3)
    n_tot = float(len(labels))
    # Diag: equal No-App vs App gradient mass, mild Unc focus [1.0, 1.12, 0.65] for APF
    raw = np.array([1.0, 1.12, 0.65], dtype=float)
    half = 0.5 * n_tot
    w_no = raw[0] * (half / (counts[0] * raw[0]))
    w_unc = raw[1] * (half / (counts[1] * raw[1] + counts[2] * raw[2]))
    w_comp = raw[2] * (half / (counts[1] * raw[1] + counts[2] * raw[2]))
    diag_sw_bin = [float(n_tot / (2.0 * counts[0])), float(n_tot / (2.0 * (counts[1] + counts[2])))]
    sev_pw_all = float(np.sqrt((counts[0] + counts[1]) / max(counts[2], 1)))

    _, c_bin, c_msk = encode_pm3_concepts(train_df, unrecorded_smoothing=0.05)
    pos = (c_bin * c_msk).sum(0)
    neg = ((1.0 - c_bin) * c_msk).sum(0)
    c_w = np.clip(np.divide(neg, pos, out=np.ones_like(pos), where=(pos > 0) & (neg > 0)), 0.5, 4.0)
    return {
        "diag_focal_w3": [float(w_no), float(w_unc), float(w_comp)],
        "diag_sw_bin": diag_sw_bin,
        "sev_pos_weight_all": sev_pw_all,
        "concept_pos_weight": c_w.tolist(),
    }

def _diag_focal_bce(logit: torch.Tensor, raw_lbl: int, weights: dict, smoothing: float, gamma: float) -> torch.Tensor:
    y = 1.0 if raw_lbl > 0 else 0.0
    ys = y * (1.0 - smoothing) + 0.5 * smoothing
    bce = F.binary_cross_entropy_with_logits(logit.float().view(()), torch.tensor(ys, device=logit.device, dtype=torch.float32), reduction="none")
    p = logit.float().sigmoid()
    pt = p if y > 0.5 else (1.0 - p)
    return float(weights["diag_focal_w3"][raw_lbl]) * ((1.0 - pt.clamp(1e-5, 1.0 - 1e-5)) ** gamma) * bce

def _sev_bce_all(logit: torch.Tensor, raw_lbl: int, pw_val: float, smoothing: float) -> torch.Tensor:
    y = 1.0 if raw_lbl == 2 else 0.0
    ys = y * (1.0 - smoothing) + 0.5 * smoothing
    pw = torch.tensor([pw_val], device=logit.device, dtype=torch.float32)
    return F.binary_cross_entropy_with_logits(logit.float().view(-1), torch.tensor([ys], device=logit.device, dtype=torch.float32), pos_weight=pw, reduction="sum")

def has_live_grads(opt: torch.optim.Optimizer) -> bool:
    return any(p.grad is not None for g in opt.param_groups for p in g["params"])

def run_pm3_joint_binary_cv(modality: str, epochs: int = 26, patience: int = 7, batch_size: int = 16, seed: int = 1) -> Tuple[dict, dict]:
    dir_diag = OUT_ROOT / f"pm3_joint_bin_{modality}_binary_diag"
    dir_sev = OUT_ROOT / f"pm3_joint_bin_{modality}_binary_sev"
    if SKIP_IF_DONE:
        ex_d, ex_s = _load_saved_binary_run(dir_diag), _load_saved_binary_run(dir_sev)
        if ex_d is not None and ex_s is not None:
            print(f"[SKIP] pm3_joint_bin_{modality}", flush=True)
            return ex_d, ex_s

    oof_d_parts, oof_s_parts, fold_rows_d, fold_rows_s = [], [], [], []
    for fold in folds:
        fk = fold["fold"]
        fc_d = dir_diag / f"fold_{fk}_val_preds.csv"
        fc_s = dir_sev / f"fold_{fk}_val_preds.csv"
        fm_d = dir_diag / f"fold_{fk}_metrics.json"
        fm_s = dir_sev / f"fold_{fk}_metrics.json"
        if SKIP_IF_DONE and fc_d.is_file() and fc_s.is_file() and fm_d.is_file() and fm_s.is_file():
            oof_d_parts.append(pd.read_csv(fc_d))
            oof_s_parts.append(pd.read_csv(fc_s))
            fold_rows_d.append(json.loads(fm_d.read_text(encoding="utf-8")))
            fold_rows_s.append(json.loads(fm_s.read_text(encoding="utf-8")))
            print(f"[RESUME] pm3_joint_bin_{modality} fold {fk}", flush=True)
            continue

        set_seed(seed + fk)
        tr_df = cohort.loc[fold["train"]]
        clin = ClinicalPipeline().fit(tr_df)
        weights = compute_pm3_bin_weights(tr_df)
        ds_tr = PM3PatientBagDataset(list(fold["train"]), cohort, clin, bags_all, image_cache_clean, train=True, include_images=True)
        ds_va = PM3PatientBagDataset(list(fold["val"]), cohort, clin, bags_all, image_cache_clean, train=False, include_images=True)
        tr_loader = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, collate_fn=collate_patients, generator=torch.Generator().manual_seed(seed + fk))
        va_loader = DataLoader(ds_va, batch_size=batch_size, shuffle=False, collate_fn=collate_patients)

        model = PM3JointBinaryNet(modality=modality).to(DEVICE)
        model.set_stage7_frozen(True)
        opt = torch.optim.AdamW(model.parameter_groups())
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=22, eta_min=1e-6)
        scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
        pw_c = torch.tensor(weights["concept_pos_weight"], device=DEVICE, dtype=torch.float32)

        best_d, best_s = -float("inf"), -float("inf")
        best_ep_d, best_ep_s, no_imp_d, no_imp_s = 0, 0, 0, 0
        min_ld, min_ls = float("inf"), float("inf")
        best_state_d = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_state_s = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        for ep in range(1, epochs + 1):
            if ep == 4:
                model.set_stage7_frozen(False)
            t0 = time.perf_counter()
            model.train()
            for items in tr_loader:
                n_b = len(items)
                n_con = max(sum(float(it["concept_mask"].sum()) for it in items), 1.0)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                    all_imgs = torch.cat([it["images"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                    all_msks = torch.cat([it["pixel_masks"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                    h_all = model.encode_chunked(all_imgs, all_msks)
                    h_splits = torch.split(h_all, [len(it["images"]) for it in items], dim=0)
                    bloss = torch.tensor(0.0, device=DEVICE, dtype=torch.float32)
                    for it, hi in zip(items, h_splits):
                        lbl = int(it["label"])
                        yd, ys_lbl = (1.0 if lbl > 0 else 0.0), (1.0 if lbl == 2 else 0.0)
                        tab_i = it["tabular"].to(DEVICE, non_blocking=True) if modality == "multimodal" else None
                        out = model.forward_bag(hi, tab_i)
                        yd_s = yd * 0.97 + 0.015
                        ld = weights["diag_sw_bin"][int(yd)] * F.binary_cross_entropy_with_logits(out["logit_diag"].view(()), torch.tensor(yd_s, device=DEVICE))
                        ls = _sev_bce_all(out["logit_sev"], lbl, weights["sev_pos_weight_all"], 0.03)
                        tc = it["concepts_target"].to(DEVICE, dtype=torch.float32)
                        mc = it["concept_mask"].to(DEVICE, dtype=torch.float32)
                        lc = (F.binary_cross_entropy_with_logits(out["concept_logits"], tc, pos_weight=pw_c, reduction="none") * mc).sum()
                        loss_i = (ld + ls) / n_b + 0.30 * (lc / n_con)
                        if modality == "multimodal":
                            ad = weights["diag_sw_bin"][int(yd)] * F.binary_cross_entropy_with_logits(out["aux_diag"].view(()), torch.tensor(yd_s, device=DEVICE))
                            as_ = _sev_bce_all(out["aux_sev"], lbl, weights["sev_pos_weight_all"], 0.03)
                            loss_i = loss_i + 0.45 * 0.5 * (ad + as_) / n_b
                        bloss = bloss + loss_i
                scaler.scale(bloss).backward()
                if has_live_grads(opt):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.5)
                    scaler.step(opt)
                scaler.update()
            if ep > 3 and ep <= 25:
                sch.step()

            model.eval()
            vp_d, vp_s, vy_d, vy_s, vld_l, vls_l = [], [], [], [], [], []
            with torch.inference_mode():
                for items in va_loader:
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                        all_imgs = torch.cat([it["images"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                        all_msks = torch.cat([it["pixel_masks"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                        h_all = model.encode_chunked(all_imgs, all_msks)
                        h_splits = torch.split(h_all, [len(it["images"]) for it in items], dim=0)
                        for it, hi in zip(items, h_splits):
                            lbl = int(it["label"])
                            yd = 1 if lbl > 0 else 0
                            tab_i = it["tabular"].to(DEVICE, non_blocking=True) if modality == "multimodal" else None
                            out = model.forward_bag(hi, tab_i)
                            vp_d.append(float(out["logit_diag"].sigmoid()))
                            vp_s.append(float(out["logit_sev"].sigmoid()))
                            vy_d.append(yd)
                            vy_s.append(1 if lbl == 2 else 0)
                            vld_l.append(float(weights["diag_sw_bin"][yd] * F.binary_cross_entropy_with_logits(out["logit_diag"].view(()), torch.tensor(float(yd), device=DEVICE))))
                            vls_l.append(float(_sev_bce_all(out["logit_sev"], lbl, weights["sev_pos_weight_all"], 0.0)))
            auc_d, auc_s = float(roc_auc_score(vy_d, vp_d)), float(roc_auc_score(vy_s, vp_s))
            m_ld, m_ls = float(np.mean(vld_l)), float(np.mean(vls_l))
            min_ld, min_ls = min(min_ld, m_ld), min(min_ls, m_ls)
            sc_d = auc_d - 0.10 * max(0.0, m_ld - min_ld)
            sc_s = auc_s - 0.10 * max(0.0, m_ls - min_ls)
            if ep >= 3:
                if sc_d > best_d:
                    best_d, best_ep_d, no_imp_d = sc_d, ep, 0
                    best_state_d = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    no_imp_d += 1
                if sc_s > best_s:
                    best_s, best_ep_s, no_imp_s = sc_s, ep, 0
                    best_state_s = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    no_imp_s += 1
            if ep in (1, epochs) or ep % 5 == 0 or (no_imp_d >= patience and no_imp_s >= patience):
                print(f"pm3_joint_bin_{modality} F{fk} ep {ep:02d}/{epochs} | AUCD={auc_d:.4f}(b={best_ep_d}) AUCS={auc_s:.4f}(b={best_ep_s}) | {time.perf_counter()-t0:.1f}s", flush=True)
            if ep > 3 and no_imp_d >= patience and no_imp_s >= patience:
                break

        def eval_state(state_cpu, which):
            model.load_state_dict({k: v.to(DEVICE) for k, v in state_cpu.items()}, strict=True)
            model.eval()
            rows = []
            with torch.inference_mode():
                for items in va_loader:
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
                        all_imgs = torch.cat([it["images"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                        all_msks = torch.cat([it["pixel_masks"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                        h_all = model.encode_chunked(all_imgs, all_msks)
                        h_splits = torch.split(h_all, [len(it["images"]) for it in items], dim=0)
                        for it, hi in zip(items, h_splits):
                            tab_i = it["tabular"].to(DEVICE, non_blocking=True) if modality == "multimodal" else None
                            out = model.forward_bag(hi, tab_i)
                            p = float(out["logit_diag"].sigmoid()) if which == "diag" else float(out["logit_sev"].sigmoid())
                            lbl = int(it["label"])
                            y_ev = (1 if lbl > 0 else 0) if which == "diag" else (1 if lbl == 2 else 0)
                            rows.append({"patient_id": int(it["patient_id"]), "raw_label": lbl, "label": y_ev, "prob": p, "p_0": 1.0 - p, "p_1": p, "pred": int(p >= 0.50), "n_images": len(it["image_ids"]), "fold": fk})
            return pd.DataFrame(rows)

        df_d = eval_state(best_state_d, "diag")
        df_s = eval_state(best_state_s, "sev")
        md = {"fold": fk, "best_epoch": best_ep_d, **{k: v for k, v in compute_binary_metrics(df_d["label"], df_d["prob"], df_d["pred"]).items() if np.isscalar(v)}}
        ms = {"fold": fk, "best_epoch": best_ep_s, **{k: v for k, v in compute_binary_metrics(df_s["label"], df_s["prob"], df_s["pred"]).items() if np.isscalar(v)}}
        dir_diag.mkdir(parents=True, exist_ok=True); dir_sev.mkdir(parents=True, exist_ok=True)
        df_d.to_csv(fc_d, index=False); df_s.to_csv(fc_s, index=False)
        save_json(fm_d, md); save_json(fm_s, ms)
        oof_d_parts.append(df_d); oof_s_parts.append(df_s)
        fold_rows_d.append(md); fold_rows_s.append(ms)
        del model, opt, sch, scaler, ds_tr, ds_va
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    res_d = save_binary_run_artifacts(dir_diag, f"pm3_joint_bin_{modality}_binary_diag", pd.concat(oof_d_parts, ignore_index=True), fold_rows_d, {"model": "PM3JointBinary", "modality": modality, "task": "binary_diag"})
    res_s = save_binary_run_artifacts(dir_sev, f"pm3_joint_bin_{modality}_binary_sev", pd.concat(oof_s_parts, ignore_index=True), fold_rows_s, {"model": "PM3JointBinary", "modality": modality, "task": "binary_sev"})
    return res_d, res_s

def run_pm3_apf09_binary_cv(modality: str, cfg: Optional[APFBinConfig] = None) -> Tuple[dict, dict]:
    cfg = cfg or APFBinConfig()
    dir_diag = OUT_ROOT / f"pm3_apf09_bin_{modality}_binary_diag"
    dir_sev = OUT_ROOT / f"pm3_apf09_bin_{modality}_binary_sev"
    if SKIP_IF_DONE:
        ex_d, ex_s = _load_saved_binary_run(dir_diag), _load_saved_binary_run(dir_sev)
        if ex_d is not None and ex_s is not None:
            print(f"[SKIP] pm3_apf09_bin_{modality}", flush=True)
            return ex_d, ex_s

    oof_d_parts, oof_s_parts, fold_rows_d, fold_rows_s = [], [], [], []
    for fold in folds:
        fk = fold["fold"]
        fc_d = dir_diag / f"fold_{fk}_val_preds.csv"
        fc_s = dir_sev / f"fold_{fk}_val_preds.csv"
        fm_d = dir_diag / f"fold_{fk}_metrics.json"
        fm_s = dir_sev / f"fold_{fk}_metrics.json"
        if SKIP_IF_DONE and fc_d.is_file() and fc_s.is_file() and fm_d.is_file() and fm_s.is_file():
            oof_d_parts.append(pd.read_csv(fc_d)); oof_s_parts.append(pd.read_csv(fc_s))
            fold_rows_d.append(json.loads(fm_d.read_text(encoding="utf-8")))
            fold_rows_s.append(json.loads(fm_s.read_text(encoding="utf-8")))
            print(f"[RESUME] pm3_apf09_bin_{modality} fold {fk}", flush=True)
            continue

        seed = cfg.seed + fk
        set_seed(seed)
        rng = np.random.default_rng(seed)
        tr_df = cohort.loc[fold["train"]]
        clin = ClinicalPipeline().fit(tr_df)
        weights = compute_pm3_bin_weights(tr_df)
        ds_tr = PM3PatientBagDataset(list(fold["train"]), cohort, clin, bags_all, image_cache_clean, train=True, include_images=True)
        ds_va = PM3PatientBagDataset(list(fold["val"]), cohort, clin, bags_all, image_cache_clean, train=False, include_images=True)
        tr_loader = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_patients, generator=torch.Generator().manual_seed(seed))
        va_loader = DataLoader(ds_va, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_patients)

        model = PM3APF09BinaryNet(modality=modality, cfg=cfg).to(DEVICE)
        model.set_stage7_frozen(True)
        opt_d = torch.optim.AdamW(model.branch_diag.parameter_groups(cfg))
        opt_s = torch.optim.AdamW(model.branch_sev.parameter_groups(cfg))
        sch_d = torch.optim.lr_scheduler.CosineAnnealingLR(opt_d, T_max=cfg.scheduler_t_max, eta_min=cfg.scheduler_eta_min)
        sch_s = torch.optim.lr_scheduler.CosineAnnealingLR(opt_s, T_max=cfg.scheduler_t_max, eta_min=cfg.scheduler_eta_min)
        ema_d, ema_s = BranchEMA(model.branch_diag, cfg.ema_decay), BranchEMA(model.branch_sev, cfg.ema_decay)
        scaler = torch.amp.GradScaler("cuda", enabled=(cfg.amp_fp16 and DEVICE.type == "cuda"))
        pw_d = torch.tensor([weights["concept_pos_weight"][k] for k in DIAG_CONCEPT_IDX], device=DEVICE, dtype=torch.float32)
        pw_s = torch.tensor([weights["concept_pos_weight"][k] for k in COMP_CONCEPT_IDX], device=DEVICE, dtype=torch.float32)

        diag_frozen, sev_frozen = False, False
        best_sd, best_ss = -float("inf"), -float("inf")
        best_ep_d, best_ep_s, no_imp_d, no_imp_s = 0, 0, 0, 0
        min_ld, min_ls = float("inf"), float("inf")
        best_state_d = {k: v.detach().cpu().clone() for k, v in model.branch_diag.state_dict().items()}
        best_state_s = {k: v.detach().cpu().clone() for k, v in model.branch_sev.state_dict().items()}

        for ep in range(1, cfg.epochs + 1):
            if cfg.warmup_freeze_epochs > 0 and ep == cfg.warmup_freeze_epochs + 1:
                model.set_stage7_frozen(False)
            t0 = time.perf_counter()
            model.train()
            for items in tr_loader:
                eff_items = []
                for it in items:
                    imgs, msks = it["images"], it["pixel_masks"]
                    if len(imgs) >= 3 and float(rng.random()) < cfg.bag_view_drop_prob:
                        keep_idx = sorted(rng.choice(len(imgs), size=len(imgs) - 1, replace=False).tolist())
                        imgs, msks = imgs[keep_idx], msks[keep_idx]
                    eff_items.append({**it, "images": imgs, "pixel_masks": msks})

                n_b = len(eff_items)
                n_cd = max(sum(float(it["concept_mask"][DIAG_CONCEPT_IDX].sum()) for it in eff_items), 1.0)
                n_cs = max(sum(float(it["concept_mask"][COMP_CONCEPT_IDX].sum()) for it in eff_items), 1.0)
                if not diag_frozen:
                    opt_d.zero_grad(set_to_none=True)
                if not sev_frozen:
                    opt_s.zero_grad(set_to_none=True)

                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(cfg.amp_fp16 and DEVICE.type == "cuda")):
                    all_imgs = torch.cat([it["images"] for it in eff_items], dim=0).to(DEVICE, non_blocking=True)
                    all_msks = torch.cat([it["pixel_masks"] for it in eff_items], dim=0).to(DEVICE, non_blocking=True)
                    hd_all, hs_all = model.encode_both_chunked(all_imgs, all_msks)
                    lens = [len(it["images"]) for it in eff_items]
                    hd_splits, hs_splits = torch.split(hd_all, lens, dim=0), torch.split(hs_all, lens, dim=0)
                    bloss = torch.tensor(0.0, device=DEVICE, dtype=torch.float32)
                    for it, hd, hs in zip(eff_items, hd_splits, hs_splits):
                        lbl = int(it["label"])
                        tab_i = it["tabular"].to(DEVICE, non_blocking=True) if modality == "multimodal" else None
                        if not diag_frozen:
                            od = model.branch_diag.forward_bag(hd, tab_i)
                            ld = _diag_focal_bce(od["logit"], lbl, weights, cfg.diag_smoothing, cfg.diag_focal_gamma)
                            td = it["concepts_target"][DIAG_CONCEPT_IDX].to(DEVICE, dtype=torch.float32)
                            md = it["concept_mask"][DIAG_CONCEPT_IDX].to(DEVICE, dtype=torch.float32)
                            lcd = (F.binary_cross_entropy_with_logits(od["concept_logits"], td, pos_weight=pw_d, reduction="none") * md).sum()
                            bloss = bloss + (ld / n_b) + cfg.w_concept_diag * (lcd / n_cd)
                            if modality == "multimodal":
                                lad = _diag_focal_bce(od["aux_img_logit"], lbl, weights, cfg.diag_smoothing, cfg.diag_focal_gamma)
                                bloss = bloss + cfg.w_aux_diag * (lad / n_b)
                        if not sev_frozen:
                            os_ = model.branch_sev.forward_bag(hs, tab_i)
                            ls = _sev_bce_all(os_["logit"], lbl, weights["sev_pos_weight_all"], cfg.sev_smoothing)
                            ts = it["concepts_target"][COMP_CONCEPT_IDX].to(DEVICE, dtype=torch.float32)
                            ms = it["concept_mask"][COMP_CONCEPT_IDX].to(DEVICE, dtype=torch.float32)
                            lcs = (F.binary_cross_entropy_with_logits(os_["concept_logits"], ts, pos_weight=pw_s, reduction="none") * ms).sum()
                            bloss = bloss + (ls / n_b) + cfg.w_concept_sev * (lcs / n_cs)
                            if modality == "multimodal":
                                las = _sev_bce_all(os_["aux_img_logit"], lbl, weights["sev_pos_weight_all"], cfg.sev_smoothing)
                                bloss = bloss + cfg.w_aux_sev * (las / n_b)
                            if cfg.sev_entropy_reg > 0 and len(hs) > 1:
                                ac = os_["attention"].float()
                                ent_n = -(ac * torch.log(ac.clamp_min(1e-7))).sum() / math.log(len(hs))
                                bloss = bloss + cfg.sev_entropy_reg * (ent_n / n_b)
                scaler.scale(bloss).backward()
                if not diag_frozen and has_live_grads(opt_d):
                    scaler.unscale_(opt_d)
                    torch.nn.utils.clip_grad_norm_(model.branch_diag.parameters(), 1.2)
                    scaler.step(opt_d)
                if not sev_frozen and has_live_grads(opt_s):
                    scaler.unscale_(opt_s)
                    torch.nn.utils.clip_grad_norm_(model.branch_sev.parameters(), 1.0)
                    scaler.step(opt_s)
                scaler.update()
                if ep >= cfg.ema_start_epoch:
                    if not diag_frozen:
                        ema_d.update(model.branch_diag)
                    if not sev_frozen:
                        ema_s.update(model.branch_sev)

            if ep > cfg.warmup_freeze_epochs and ep <= cfg.warmup_freeze_epochs + cfg.scheduler_t_max:
                if not diag_frozen:
                    sch_d.step()
                if not sev_frozen:
                    sch_s.step()

            bk_d = ema_d.apply_to(model.branch_diag) if (ep >= cfg.ema_start_epoch and not diag_frozen) else None
            bk_s = ema_s.apply_to(model.branch_sev) if (ep >= cfg.ema_start_epoch and not sev_frozen) else None
            model.eval()
            vp_d, vp_s, vy_d, vy_s, vld_l, vls_l = [], [], [], [], [], []
            with torch.inference_mode():
                for items in va_loader:
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(cfg.amp_fp16 and DEVICE.type == "cuda")):
                        all_imgs = torch.cat([it["images"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                        all_msks = torch.cat([it["pixel_masks"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                        hd_all, hs_all = model.encode_both_chunked(all_imgs, all_msks)
                        lens = [len(it["images"]) for it in items]
                        for it, hd, hs in zip(items, torch.split(hd_all, lens, dim=0), torch.split(hs_all, lens, dim=0)):
                            lbl = int(it["label"])
                            tab_i = it["tabular"].to(DEVICE, non_blocking=True) if modality == "multimodal" else None
                            od = model.branch_diag.forward_bag(hd, tab_i)
                            os_ = model.branch_sev.forward_bag(hs, tab_i)
                            vp_d.append(float(od["prob"])); vp_s.append(float(os_["prob"]))
                            vy_d.append(1 if lbl > 0 else 0); vy_s.append(1 if lbl == 2 else 0)
                            vld_l.append(float(_diag_focal_bce(od["logit"], lbl, weights, cfg.diag_smoothing, cfg.diag_focal_gamma)))
                            vls_l.append(float(_sev_bce_all(os_["logit"], lbl, weights["sev_pos_weight_all"], cfg.sev_smoothing)))
            if bk_d is not None:
                ema_d.restore(model.branch_diag, bk_d)
            if bk_s is not None:
                ema_s.restore(model.branch_sev, bk_s)

            auc_d, auc_s = float(roc_auc_score(vy_d, vp_d)), float(roc_auc_score(vy_s, vp_s))
            m_ld, m_ls = float(np.mean(vld_l)), float(np.mean(vls_l))
            min_ld, min_ls = min(min_ld, m_ld), min(min_ls, m_ls)
            sc_d = auc_d - cfg.loss_guard_penalty * max(0.0, m_ld - min_ld)
            sc_s = auc_s - cfg.loss_guard_penalty * max(0.0, m_ls - min_ls)

            if ep >= cfg.warmup_freeze_epochs:
                if not diag_frozen:
                    if sc_d > best_sd:
                        best_sd, best_ep_d, no_imp_d = sc_d, ep, 0
                        bk = ema_d.apply_to(model.branch_diag) if ep >= cfg.ema_start_epoch else None
                        best_state_d = {k: v.detach().cpu().clone() for k, v in model.branch_diag.state_dict().items()}
                        if bk is not None:
                            ema_d.restore(model.branch_diag, bk)
                    else:
                        no_imp_d += 1
                if not sev_frozen:
                    if sc_s > best_ss:
                        best_ss, best_ep_s, no_imp_s = sc_s, ep, 0
                        bk = ema_s.apply_to(model.branch_sev) if ep >= cfg.ema_start_epoch else None
                        best_state_s = {k: v.detach().cpu().clone() for k, v in model.branch_sev.state_dict().items()}
                        if bk is not None:
                            ema_s.restore(model.branch_sev, bk)
                    else:
                        no_imp_s += 1

            ev = ""
            if (not sev_frozen) and no_imp_s >= cfg.patience_sev and ep >= cfg.min_freeze_epoch_sev:
                model.branch_sev.load_state_dict({k: v.to(DEVICE) for k, v in best_state_s.items()}, strict=True)
                model.branch_sev.freeze_entire_branch()
                sev_frozen = True
                ev += f"[FREEZE SEV@{ep}->b{best_ep_s}] "
            if (not diag_frozen) and no_imp_d >= cfg.patience_diag and ep >= cfg.min_freeze_epoch_diag:
                model.branch_diag.load_state_dict({k: v.to(DEVICE) for k, v in best_state_d.items()}, strict=True)
                model.branch_diag.freeze_entire_branch()
                diag_frozen = True
                ev += f"[FREEZE DIAG@{ep}->b{best_ep_d}] "

            if ep in (1, cfg.epochs) or ep % 5 == 0 or ev or (diag_frozen and sev_frozen):
                print(f"pm3_apf09_bin_{modality} F{fk} ep {ep:02d}/{cfg.epochs} | AUCD={auc_d:.4f}(b={best_ep_d}) AUCS={auc_s:.4f}(b={best_ep_s}) | {time.perf_counter()-t0:.1f}s {ev}", flush=True)
            if diag_frozen and sev_frozen:
                break

        model.branch_diag.load_state_dict({k: v.to(DEVICE) for k, v in best_state_d.items()}, strict=True)
        model.branch_sev.load_state_dict({k: v.to(DEVICE) for k, v in best_state_s.items()}, strict=True)
        model.eval()
        rows_d, rows_s = [], []
        with torch.inference_mode():
            for items in va_loader:
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(cfg.amp_fp16 and DEVICE.type == "cuda")):
                    all_imgs = torch.cat([it["images"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                    all_msks = torch.cat([it["pixel_masks"] for it in items], dim=0).to(DEVICE, non_blocking=True)
                    hd_all, hs_all = model.encode_both_chunked(all_imgs, all_msks)
                    lens = [len(it["images"]) for it in items]
                    for it, hd, hs in zip(items, torch.split(hd_all, lens, dim=0), torch.split(hs_all, lens, dim=0)):
                        lbl = int(it["label"])
                        tab_i = it["tabular"].to(DEVICE, non_blocking=True) if modality == "multimodal" else None
                        pd_v = float(model.branch_diag.forward_bag(hd, tab_i)["prob"])
                        ps_v = float(model.branch_sev.forward_bag(hs, tab_i)["prob"])
                        rows_d.append({"patient_id": int(it["patient_id"]), "raw_label": lbl, "label": int(lbl > 0), "prob": pd_v, "p_0": 1.0 - pd_v, "p_1": pd_v, "pred": int(pd_v >= 0.50), "n_images": len(it["image_ids"]), "fold": fk})
                        rows_s.append({"patient_id": int(it["patient_id"]), "raw_label": lbl, "label": int(lbl == 2), "prob": ps_v, "p_0": 1.0 - ps_v, "p_1": ps_v, "pred": int(ps_v >= 0.50), "n_images": len(it["image_ids"]), "fold": fk})
        df_d, df_s = pd.DataFrame(rows_d), pd.DataFrame(rows_s)
        md = {"fold": fk, "best_epoch": best_ep_d, **{k: v for k, v in compute_binary_metrics(df_d["label"], df_d["prob"], df_d["pred"]).items() if np.isscalar(v)}}
        ms = {"fold": fk, "best_epoch": best_ep_s, **{k: v for k, v in compute_binary_metrics(df_s["label"], df_s["prob"], df_s["pred"]).items() if np.isscalar(v)}}
        dir_diag.mkdir(parents=True, exist_ok=True); dir_sev.mkdir(parents=True, exist_ok=True)
        df_d.to_csv(fc_d, index=False); df_s.to_csv(fc_s, index=False)
        save_json(fm_d, md); save_json(fm_s, ms)
        oof_d_parts.append(df_d); oof_s_parts.append(df_s)
        fold_rows_d.append(md); fold_rows_s.append(ms)
        del model, opt_d, opt_s, sch_d, sch_s, scaler, ds_tr, ds_va
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    res_d = save_binary_run_artifacts(dir_diag, f"pm3_apf09_bin_{modality}_binary_diag", pd.concat(oof_d_parts, ignore_index=True), fold_rows_d, {"model": "PM3APF09Binary", "modality": modality, "task": "binary_diag"})
    res_s = save_binary_run_artifacts(dir_sev, f"pm3_apf09_bin_{modality}_binary_sev", pd.concat(oof_s_parts, ignore_index=True), fold_rows_s, {"model": "PM3APF09Binary", "modality": modality, "task": "binary_sev"})
    return res_d, res_s

pm3_bin_results = {}
if RUN_PM3_BINARY:
    for mod in ("image_only", "multimodal"):
        pm3_bin_results[f"pm3_joint_bin_{mod}"] = run_pm3_joint_binary_cv(mod, epochs=26, patience=7)
        pm3_bin_results[f"pm3_apf09_bin_{mod}"] = run_pm3_apf09_binary_cv(mod)


[SKIP] pm3_joint_bin_image_only
[SKIP] pm3_apf09_bin_image_only
[SKIP] pm3_joint_bin_multimodal
[SKIP] pm3_apf09_bin_multimodal


## 2. Replikasi Paper 1 — `MVCBM` (Marcinkevičs et al., 2024): `MVC-Mean` & `MVC-LSTM`


In [ ]:
class MVCBMNet(nn.Module):
    def __init__(self, aggregator: str = "avg", ex_dim: int = 0, t_hidden_dim: int = 5, num_concepts: int = 9):
        super().__init__()
        assert aggregator in ("avg", "lstm")
        self.aggregator = aggregator
        self.ex_dim = ex_dim
        self.num_concepts = num_concepts
        self.backbone = load_resnet18_backbone()
        self.embed_dim = 512
        self.c_rnn = nn.LSTM(self.embed_dim, self.embed_dim, num_layers=1, batch_first=True) if aggregator == "lstm" else None
        self.c_fc1 = nn.Linear(self.embed_dim, 256)
        self.c_fc2 = nn.Linear(256, 64)
        self.c_fc3 = nn.Linear(64, num_concepts)
        self.t_fc1 = nn.Linear(num_concepts + ex_dim, t_hidden_dim)
        self.t_fc2 = nn.Linear(t_hidden_dim, 1)

    def encode_views(self, all_imgs: torch.Tensor, microbatch: int = 16) -> torch.Tensor:
        return encode_resnet18_microbatched(self.backbone, all_imgs, microbatch=microbatch)

    def forward_concepts_from_view_feats(self, feat_splits: List[torch.Tensor]) -> torch.Tensor:
        pooled = []
        for f_i in feat_splits:
            if self.aggregator == "avg":
                pooled.append(f_i.float().mean(dim=0))
            else:
                _, (hn, _) = self.c_rnn(f_i.float().unsqueeze(0))
                pooled.append(hn[-1, 0])
        h = F.relu(self.c_fc1(torch.stack(pooled, dim=0)))
        return self.c_fc3(F.relu(self.c_fc2(h)))

    def forward_target(self, c_probs: torch.Tensor, ex_feat: Optional[torch.Tensor] = None) -> torch.Tensor:
        inp = torch.cat([c_probs, ex_feat], dim=1) if (self.ex_dim > 0 and ex_feat is not None) else c_probs
        return self.t_fc2(F.relu(self.t_fc1(inp))).squeeze(1)

    def forward(self, items: List[dict], device: torch.device, microbatch: int = 16) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        lengths = [len(it["images"]) for it in items]
        all_imgs = torch.cat([it["images"] for it in items], dim=0).to(device, non_blocking=True)
        feat_splits = list(torch.split(self.encode_views(all_imgs, microbatch=microbatch), lengths, dim=0))
        c_logits = self.forward_concepts_from_view_feats(feat_splits)
        c_probs = torch.sigmoid(c_logits.float())
        ex_feat = torch.stack([it["tabular"] for it in items], dim=0).to(device, non_blocking=True).float() if self.ex_dim > 0 else None
        t_logit = self.forward_target(c_probs, ex_feat)
        return c_logits, c_probs, t_logit

def compute_mvcbm_balanced_weights(y_bin: np.ndarray, c_mat: np.ndarray) -> Tuple[np.ndarray, List[np.ndarray]]:
    t_cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_bin.astype(int)).astype(np.float32)
    c_cw = []
    for j in range(c_mat.shape[1]):
        cj = c_mat[:, j].astype(int)
        w = compute_class_weight("balanced", classes=np.array([0, 1]), y=cj).astype(np.float32) if len(np.unique(cj)) == 2 else np.array([1.0, 1.0], dtype=np.float32)
        c_cw.append(w)
    return t_cw, c_cw

def mvcbm_loss_fn(c_logits, t_logit, c_true, y_true, t_cw, c_cw, alpha: float = 1.0):
    y_np = y_true.detach().cpu().numpy().astype(int)
    w_t = torch.tensor([float(t_cw[v]) for v in y_np], device=t_logit.device, dtype=torch.float32)
    t_loss = F.binary_cross_entropy_with_logits(t_logit.float(), y_true.float(), weight=w_t, reduction="mean")
    c_np = c_true.detach().cpu().numpy().astype(int)
    c_losses = [
        F.binary_cross_entropy_with_logits(
            c_logits[:, j].float(), c_true[:, j].float(),
            weight=torch.tensor([float(c_cw[j][v]) for v in c_np[:, j]], device=c_logits.device, dtype=torch.float32),
            reduction="mean",
        )
        for j in range(c_true.shape[1])
    ]
    c_loss_sum = torch.stack(c_losses).sum()
    return t_loss, c_loss_sum, t_loss + alpha * c_loss_sum

def train_mvcbm_cmodel_stage(
    aggregator: str, folds_list: List[dict], cohort_df: pd.DataFrame, bags: Dict[int, List[str]],
    cache: Dict[str, Path], stage_dir: Path, c_epochs: int = 20, batch_size: int = 16, lr: float = 1e-4, seed: int = 1, amp_fp16: bool = True,
) -> Dict[int, dict]:
    stage_dir.mkdir(parents=True, exist_ok=True)
    cached_folds = {}
    for fold in folds_list:
        fk = fold["fold"]
        fdir = stage_dir / f"fold_{fk}"
        fdir.mkdir(parents=True, exist_ok=True)
        npz_path = fdir / "c_hat_cache.npz"
        if SKIP_IF_DONE and npz_path.is_file():
            with np.load(npz_path) as d:
                cached_folds[fk] = {k: d[k].copy() for k in d.files}
            print(f"[SKIP CModel] mvc_{aggregator} fold {fk}", flush=True)
            continue

        set_seed(seed + fk)
        pipe = ClinicalPipeline().fit(cohort_df.loc[fold["train"]])
        tr_ds = PaperBagDataset(fold["train"], cohort_df, pipe, bags, cache, "binary_diag", "usconfig", train=True)
        va_ds = PaperBagDataset(fold["val"], cohort_df, pipe, bags, cache, "binary_diag", "usconfig", train=False)
        tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_patients, generator=torch.Generator().manual_seed(seed + fk))
        va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_patients)
        va_tr_loader = DataLoader(PaperBagDataset(fold["train"], cohort_df, pipe, bags, cache, "binary_diag", "usconfig", train=False), batch_size=batch_size, shuffle=False, collate_fn=collate_patients)
        _, c_cw = compute_mvcbm_balanced_weights(tr_ds.y_bin, tr_ds.concepts_9)
        t_cw_dummy = np.array([1.0, 1.0], dtype=np.float32)

        model = MVCBMNet(aggregator=aggregator, ex_dim=0, t_hidden_dim=5).to(DEVICE)
        for p in list(model.t_fc1.parameters()) + list(model.t_fc2.parameters()):
            p.requires_grad_(False)
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
        scaler = torch.amp.GradScaler("cuda", enabled=(amp_fp16 and DEVICE.type == "cuda"))

        best_val_c, best_state = float("inf"), None
        for ep in range(1, c_epochs + 1):
            t0 = time.perf_counter()
            model.train()
            for items in tr_loader:
                opt.zero_grad(set_to_none=True)
                c_true = torch.stack([it["concepts_9"] for it in items], dim=0).to(DEVICE)
                y_true = torch.tensor([it["y_bin"] for it in items], device=DEVICE, dtype=torch.float32)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(amp_fp16 and DEVICE.type == "cuda")):
                    c_logits, _, t_logit = model(items, DEVICE)
                _, c_loss_sum, _ = mvcbm_loss_fn(c_logits, t_logit, c_true, y_true, t_cw_dummy, c_cw, alpha=1.0)
                scaler.scale(c_loss_sum).backward()
                scaler.step(opt)
                scaler.update()

            model.eval()
            va_losses = []
            with torch.inference_mode():
                for items in va_loader:
                    c_true = torch.stack([it["concepts_9"] for it in items], dim=0).to(DEVICE)
                    y_true = torch.tensor([it["y_bin"] for it in items], device=DEVICE, dtype=torch.float32)
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(amp_fp16 and DEVICE.type == "cuda")):
                        c_logits, _, t_logit = model(items, DEVICE)
                    _, c_loss_sum, _ = mvcbm_loss_fn(c_logits, t_logit, c_true, y_true, t_cw_dummy, c_cw, alpha=1.0)
                    va_losses.append(float(c_loss_sum.item()))
            mean_vc = float(np.mean(va_losses))
            if mean_vc < best_val_c:
                best_val_c, best_state = mean_vc, copy.deepcopy(model.state_dict())
            if ep in (1, c_epochs) or ep % 5 == 0:
                print(f"CModel-{aggregator} | fold {fk} | ep {ep}/{c_epochs} | val_c_loss={mean_vc:.4f} | {time.perf_counter()-t0:.1f}s", flush=True)

        model.load_state_dict(best_state)
        model.eval()
        def extract_split(ldr):
            cs, raws, pids, tabs, ni = [], [], [], [], []
            with torch.inference_mode():
                for items in ldr:
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(amp_fp16 and DEVICE.type == "cuda")):
                        c_logits, _, _ = model(items, DEVICE)
                    cs.append(torch.sigmoid(c_logits.float()).cpu().numpy())
                    raws.extend([it["raw_label"] for it in items])
                    pids.extend([it["patient_id"] for it in items])
                    ni.extend([len(it["image_ids"]) for it in items])
                    tabs.append(torch.stack([it["tabular"] for it in items], dim=0).numpy())
            return np.concatenate(cs, axis=0).astype(np.float32), np.asarray(raws, dtype=int), np.asarray(pids, dtype=int), np.asarray(ni, dtype=int), np.concatenate(tabs, axis=0).astype(np.float32)

        tr_c, tr_raw, tr_pid, tr_ni, tr_tab = extract_split(va_tr_loader)
        va_c, va_raw, va_pid, va_ni, va_tab = extract_split(va_loader)
        payload = {"tr_c": tr_c, "tr_raw": tr_raw, "tr_pid": tr_pid, "tr_ni": tr_ni, "tr_tab": tr_tab, "va_c": va_c, "va_raw": va_raw, "va_pid": va_pid, "va_ni": va_ni, "va_tab": va_tab}
        np.savez_compressed(npz_path, **payload)
        cached_folds[fk] = payload
        del model, opt, scaler, tr_ds, va_ds
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return cached_folds

def fit_mvcbm_tmodel_suite(aggregator: str, cached_folds: Dict[int, dict], t_epochs: int = 30, lr: float = 1e-2, seed: int = 1) -> dict:
    results = {}
    specs = [("image_only", "binary_diag", 5), ("image_only", "binary_sev", 5), ("multimodal", "binary_diag", 16), ("multimodal", "binary_sev", 16)]
    for modality, task_mode, t_hidden in specs:
        run_name = f"mvcbm_seq_{modality}_{task_mode}_{aggregator}"
        out_dir = OUT_ROOT / run_name
        existing = _load_saved_binary_run(out_dir) if SKIP_IF_DONE else None
        if existing is not None:
            print(f"[SKIP] {run_name}", flush=True)
            results[run_name] = existing
            continue
        oof_parts, fold_rows = [], []
        ex_dim = 28 if modality == "multimodal" else 0
        for fk, d in sorted(cached_folds.items()):
            set_seed(seed + fk)
            y_tr = (d["tr_raw"] > 0).astype(np.float32) if task_mode == "binary_diag" else (d["tr_raw"] == 2).astype(np.float32)
            y_va = (d["va_raw"] > 0).astype(np.float32) if task_mode == "binary_diag" else (d["va_raw"] == 2).astype(np.float32)
            x_tr = np.hstack([d["tr_c"], d["tr_tab"]]) if ex_dim > 0 else d["tr_c"]
            x_va = np.hstack([d["va_c"], d["va_tab"]]) if ex_dim > 0 else d["va_c"]
            t_cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_tr.astype(int)).astype(np.float32)
            w_tr_t = torch.tensor([float(t_cw[int(v)]) for v in y_tr], device=DEVICE, dtype=torch.float32)
            x_tr_t, y_tr_t, x_va_t = torch.from_numpy(x_tr).to(DEVICE), torch.from_numpy(y_tr).to(DEVICE), torch.from_numpy(x_va).to(DEVICE)
            t_net = nn.Sequential(nn.Linear(x_tr.shape[1], t_hidden), nn.ReLU(), nn.Linear(t_hidden, 1)).to(DEVICE)
            opt = torch.optim.Adam(t_net.parameters(), lr=lr)
            best_auc, best_probs, best_loss = -1.0, None, float("inf")
            for ep in range(1, t_epochs + 1):
                t_net.train(); opt.zero_grad(set_to_none=True)
                loss = F.binary_cross_entropy_with_logits(t_net(x_tr_t).squeeze(1), y_tr_t, weight=w_tr_t, reduction="mean")
                loss.backward(); opt.step()
                t_net.eval()
                with torch.inference_mode():
                    va_logit = t_net(x_va_t).squeeze(1)
                    va_prob = torch.sigmoid(va_logit).cpu().numpy()
                    va_loss = float(F.binary_cross_entropy_with_logits(va_logit, torch.from_numpy(y_va).to(DEVICE)).item())
                    va_auc = float(roc_auc_score(y_va.astype(int), va_prob))
                if (va_auc, -va_loss) > (best_auc, -best_loss):
                    best_auc, best_loss, best_probs = va_auc, va_loss, va_prob.copy()
            m = compute_binary_metrics(y_va.astype(int), best_probs, (best_probs >= 0.50).astype(int))
            fold_rows.append({"fold": fk, **{k: v for k, v in m.items() if np.isscalar(v)}})
            oof_parts.append(pd.DataFrame({"patient_id": d["va_pid"], "raw_label": d["va_raw"], "label": y_va.astype(int), "prob": best_probs.astype(float), "p_0": 1.0 - best_probs.astype(float), "p_1": best_probs.astype(float), "pred": (best_probs >= 0.50).astype(int), "n_images": d["va_ni"], "fold": fk}))
        results[run_name] = save_binary_run_artifacts(out_dir, run_name, pd.concat(oof_parts, ignore_index=True), fold_rows, {"paper": "Marcinkevics2024_Sequential", "aggregator": aggregator, "modality": modality, "task_mode": task_mode})
    return results

def run_mvcbm_joint_one(aggregator: str, modality: str, task_mode: str, folds_list: List[dict], cohort_df: pd.DataFrame, bags: Dict[int, List[str]], cache: Dict[str, Path], epochs: int = 30, patience: int = 10, batch_size: int = 16, lr: float = 1e-4, alpha: float = 1.0, seed: int = 1, amp_fp16: bool = True) -> dict:
    run_name = f"mvcbm_joint_{modality}_{task_mode}_{aggregator}"
    out_dir = OUT_ROOT / run_name
    existing = _load_saved_binary_run(out_dir) if SKIP_IF_DONE else None
    if existing is not None:
        print(f"[SKIP] {run_name}", flush=True)
        return existing
    ex_dim, t_hidden = (28, 16) if modality == "multimodal" else (0, 5)
    oof_parts, fold_rows = [], []
    for fold in folds_list:
        fk = fold["fold"]
        fc, fm = out_dir / f"fold_{fk}_val_preds.csv", out_dir / f"fold_{fk}_metrics.json"
        if SKIP_IF_DONE and fc.is_file() and fm.is_file():
            oof_parts.append(pd.read_csv(fc)); fold_rows.append(json.loads(fm.read_text(encoding="utf-8"))); continue
        set_seed(seed + fk)
        pipe = ClinicalPipeline().fit(cohort_df.loc[fold["train"]])
        tr_ds = PaperBagDataset(fold["train"], cohort_df, pipe, bags, cache, task_mode, "usconfig", train=True)
        va_ds = PaperBagDataset(fold["val"], cohort_df, pipe, bags, cache, task_mode, "usconfig", train=False)
        tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_patients, generator=torch.Generator().manual_seed(seed + fk))
        va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_patients)
        t_cw, c_cw = compute_mvcbm_balanced_weights(tr_ds.y_bin, tr_ds.concepts_9)
        model = MVCBMNet(aggregator=aggregator, ex_dim=ex_dim, t_hidden_dim=t_hidden).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        scaler = torch.amp.GradScaler("cuda", enabled=(amp_fp16 and DEVICE.type == "cuda"))
        best_score, best_state, no_imp, best_ep = (-1.0, -float("inf")), None, 0, 0
        for ep in range(1, epochs + 1):
            t0 = time.perf_counter(); model.train()
            for items in tr_loader:
                opt.zero_grad(set_to_none=True)
                c_true = torch.stack([it["concepts_9"] for it in items], dim=0).to(DEVICE)
                y_true = torch.tensor([it["y_bin"] for it in items], device=DEVICE, dtype=torch.float32)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(amp_fp16 and DEVICE.type == "cuda")):
                    c_logits, _, t_logit = model(items, DEVICE)
                _, _, tot_loss = mvcbm_loss_fn(c_logits, t_logit, c_true, y_true, t_cw, c_cw, alpha=alpha)
                scaler.scale(tot_loss).backward(); scaler.step(opt); scaler.update()
            model.eval(); va_probs, va_ys, va_losses = [], [], []
            with torch.inference_mode():
                for items in va_loader:
                    c_true = torch.stack([it["concepts_9"] for it in items], dim=0).to(DEVICE)
                    y_true = torch.tensor([it["y_bin"] for it in items], device=DEVICE, dtype=torch.float32)
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(amp_fp16 and DEVICE.type == "cuda")):
                        c_logits, _, t_logit = model(items, DEVICE)
                    t_loss, _, _ = mvcbm_loss_fn(c_logits, t_logit, c_true, y_true, t_cw, c_cw, alpha=alpha)
                    va_probs.extend(torch.sigmoid(t_logit.float()).cpu().tolist()); va_ys.extend([int(it["y_bin"]) for it in items]); va_losses.append(float(t_loss.item()))
            cand = (float(roc_auc_score(va_ys, va_probs)), -float(np.mean(va_losses)))
            if cand > best_score:
                best_score, best_state, no_imp, best_ep = cand, copy.deepcopy(model.state_dict()), 0, ep
            else:
                no_imp += 1
            if ep in (1, epochs) or ep % 5 == 0 or no_imp >= patience:
                print(f"{run_name} F{fk} ep {ep}/{epochs} | val_auc={cand[0]:.4f} | {time.perf_counter()-t0:.1f}s", flush=True)
            if no_imp >= patience:
                break
        model.load_state_dict(best_state); model.eval(); rows = []
        with torch.inference_mode():
            for items in va_loader:
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(amp_fp16 and DEVICE.type == "cuda")):
                    _, _, t_logit = model(items, DEVICE)
                for it, p in zip(items, torch.sigmoid(t_logit.float()).cpu().numpy()):
                    rows.append({"patient_id": int(it["patient_id"]), "raw_label": int(it["raw_label"]), "label": int(it["y_bin"]), "prob": float(p), "p_0": 1.0 - float(p), "p_1": float(p), "pred": int(p >= 0.50), "n_images": len(it["image_ids"]), "fold": fk})
        f_df = pd.DataFrame(rows)
        m = compute_binary_metrics(f_df["label"], f_df["prob"], f_df["pred"])
        m_row = {"fold": fk, "best_epoch": best_ep, **{k: v for k, v in m.items() if np.isscalar(v)}}
        out_dir.mkdir(parents=True, exist_ok=True); f_df.to_csv(fc, index=False); save_json(fm, m_row)
        oof_parts.append(f_df); fold_rows.append(m_row)
        del model, opt, scaler, tr_ds, va_ds; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return save_binary_run_artifacts(out_dir, run_name, pd.concat(oof_parts, ignore_index=True), fold_rows, {"paper": "Marcinkevics2024_Joint", "aggregator": aggregator, "modality": modality, "task_mode": task_mode})

mvcbm_results = {}
if RUN_MVCBM_SEQ:
    for agg in ("avg", "lstm"):
        c_folds = train_mvcbm_cmodel_stage(agg, folds, cohort, bags_all, image_cache_clean, OUT_ROOT / f"_cmodel_cache_{agg}", c_epochs=20)
        mvcbm_results.update(fit_mvcbm_tmodel_suite(agg, c_folds, t_epochs=30))
if RUN_MVCBM_JOINT:
    for modality in ("image_only", "multimodal"):
        for task_mode in ("binary_diag", "binary_sev"):
            for agg in ("avg", "lstm"):
                mvcbm_results[f"mvcbm_joint_{modality}_{task_mode}_{agg}"] = run_mvcbm_joint_one(agg, modality, task_mode, folds, cohort, bags_all, image_cache_clean, epochs=30, patience=10)


[SKIP CModel] mvc_avg fold 1
CModel-avg | fold 2 | ep 1/20 | val_c_loss=6.3300 | 36.1s
CModel-avg | fold 2 | ep 5/20 | val_c_loss=6.3354 | 18.0s
CModel-avg | fold 2 | ep 10/20 | val_c_loss=6.3248 | 16.9s
CModel-avg | fold 2 | ep 15/20 | val_c_loss=6.3294 | 18.3s
CModel-avg | fold 2 | ep 20/20 | val_c_loss=6.3119 | 19.7s
CModel-avg | fold 3 | ep 1/20 | val_c_loss=6.2416 | 85.3s


## 3. Replikasi Paper 2 — Barthwal et al. (2026): `ResNet-18 + TabNet + Cross-Attention` (`leaky_53` vs `clean_28`)


In [ ]:
def sparsemax(logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
    z = logits.float()
    z_sorted, _ = torch.sort(z, dim=dim, descending=True)
    k = torch.arange(1, z.size(dim) + 1, device=z.device, dtype=z.dtype)
    view_shape = [1] * z.dim(); view_shape[dim] = -1
    k = k.view(view_shape)
    cumsum = torch.cumsum(z_sorted, dim=dim)
    support = (1.0 + k * z_sorted) > cumsum
    k_z = support.sum(dim=dim, keepdim=True).clamp(min=1)
    tau = (torch.gather(cumsum, dim, k_z - 1) - 1.0) / k_z.to(z.dtype)
    return torch.clamp(z - tau, min=0.0)

class GatedFeatureBlock(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.fc = nn.Linear(in_dim, 2 * out_dim, bias=False)
        self.norm = nn.LayerNorm(2 * out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a, b = self.norm(self.fc(x)).chunk(2, dim=-1)
        return a * torch.sigmoid(b)

class TabNetFeatureTransformer(nn.Module):
    def __init__(self, shared_blocks: nn.ModuleList, step_blocks: nn.ModuleList):
        super().__init__()
        self.shared_blocks = shared_blocks
        self.step_blocks = step_blocks
        self.scale = math.sqrt(0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.shared_blocks[0](x)
        for blk in self.shared_blocks[1:]:
            h = (h + blk(h)) * self.scale
        for blk in self.step_blocks:
            h = (h + blk(h)) * self.scale
        return h

class TabNetEncoder(nn.Module):
    def __init__(self, in_dim: int, n_d: int = 64, n_a: int = 64, n_steps: int = 5, gamma: float = 1.3):
        super().__init__()
        self.in_dim, self.n_d, self.n_a, self.n_steps, self.gamma = in_dim, n_d, n_a, n_steps, gamma
        self.in_norm = nn.LayerNorm(in_dim)
        dim_tot = n_d + n_a
        shared_blocks = nn.ModuleList([GatedFeatureBlock(in_dim, dim_tot), GatedFeatureBlock(dim_tot, dim_tot)])
        self.initial_splitter = TabNetFeatureTransformer(shared_blocks, nn.ModuleList())
        self.step_transformers = nn.ModuleList()
        self.attn_fcs = nn.ModuleList()
        self.attn_norms = nn.ModuleList()
        for _ in range(n_steps):
            self.step_transformers.append(TabNetFeatureTransformer(shared_blocks, nn.ModuleList([GatedFeatureBlock(dim_tot, dim_tot), GatedFeatureBlock(dim_tot, dim_tot)])))
            self.attn_fcs.append(nn.Linear(n_a, in_dim, bias=False))
            self.attn_norms.append(nn.LayerNorm(in_dim))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x = self.in_norm(x.float())
        prior = torch.ones_like(x)
        att = self.initial_splitter(x)[:, self.n_d:]
        step_outs = []
        m_explain = torch.zeros_like(x)
        m_loss = torch.tensor(0.0, device=x.device, dtype=torch.float32)
        for step in range(self.n_steps):
            mask = sparsemax(prior * self.attn_norms[step](self.attn_fcs[step](att)), dim=-1)
            prior = prior * (self.gamma - mask)
            m_loss = m_loss + (mask * torch.log(mask.clamp_min(1e-8))).sum(dim=-1).mean() / self.n_steps
            h_step = self.step_transformers[step](mask * x)
            d_step = F.relu(h_step[:, :self.n_d])
            att = h_step[:, self.n_d:]
            step_outs.append(d_step)
            m_explain = m_explain + d_step.sum(dim=-1, keepdim=True) * mask
        return torch.stack(step_outs, dim=1), m_loss, m_explain

class BarthwalMultimodalNet(nn.Module):
    def __init__(self, tab_dim: int, d_model: int = 64, n_steps: int = 5):
        super().__init__()
        self.d_model = d_model
        self.backbone = load_resnet18_backbone()
        self.img_proj = nn.Sequential(nn.Linear(512, d_model), nn.LayerNorm(d_model))
        self.tabnet = TabNetEncoder(in_dim=tab_dim, n_d=d_model, n_a=d_model, n_steps=n_steps, gamma=1.3)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.fuse_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.30),
            nn.Linear(64, 1),
        )

    def forward(self, items: List[dict], device: torch.device, microbatch: int = 16) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        lengths = [len(it["images"]) for it in items]
        all_imgs = torch.cat([it["images"] for it in items], dim=0).to(device, non_blocking=True)
        all_feats = encode_resnet18_microbatched(self.backbone, all_imgs, microbatch=microbatch)
        f_img_raw = torch.stack([f_i.float().mean(dim=0) for f_i in torch.split(all_feats, lengths, dim=0)], dim=0)
        f_img = self.img_proj(f_img_raw).unsqueeze(1)
        x_tab = torch.stack([it["tabular"] for it in items], dim=0).to(device, non_blocking=True).float()
        f_tab_tokens, m_loss, m_explain = self.tabnet(x_tab)
        Q, K, V = self.W_q(f_img.float()), self.W_k(f_tab_tokens.float()), self.W_v(f_tab_tokens.float())
        attn_w = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model), dim=-1)
        f_fused = self.fuse_norm(torch.matmul(attn_w, V).squeeze(1) + f_img.squeeze(1))
        return self.classifier(f_fused).squeeze(1), m_loss, m_explain

def run_barthwal_cv(
    regime: str, task_mode: str, folds_list: List[dict], cohort_df: pd.DataFrame, bags: Dict[int, List[str]],
    clean_cache: Dict[str, Path], barthwal_cache: Dict[str, Path], epochs: int = 30, patience: int = 9,
    batch_size: int = 16, lr_backbone: float = 1e-4, lr_tab_fusion: float = 1e-3, lambda_sparse: float = 1e-3, seed: int = 1, amp_fp16: bool = True,
) -> dict:
    assert regime in ("paper_leaky53", "our_clean28")
    run_name = f"barthwal_{regime}_{task_mode}"
    out_dir = OUT_ROOT / run_name
    existing = _load_saved_binary_run(out_dir) if SKIP_IF_DONE else None
    if existing is not None:
        print(f"[SKIP] {run_name}", flush=True)
        imp_p = out_dir / "tabnet_feature_importance.csv"
        if imp_p.is_file():
            existing["importance"] = pd.read_csv(imp_p)
        return existing

    is_paper = (regime == "paper_leaky53")
    pipe_cls = LeakyClinicalPipeline if is_paper else ClinicalPipeline
    img_regime = "barthwal_224" if is_paper else "usconfig"
    cache = barthwal_cache if is_paper else clean_cache
    oof_parts, fold_rows, imp_accum = [], [], []
    col_names_ref = None

    for fold in folds_list:
        fk = fold["fold"]
        fold_csv, fold_json, fold_imp_npy = out_dir / f"fold_{fk}_val_preds.csv", out_dir / f"fold_{fk}_metrics.json", out_dir / f"fold_{fk}_imp.npy"
        pipe_tmp = pipe_cls().fit(cohort_df.loc[fold["train"]])
        col_names_ref = list(pipe_tmp.columns_)
        if SKIP_IF_DONE and fold_csv.is_file() and fold_json.is_file() and fold_imp_npy.is_file():
            oof_parts.append(pd.read_csv(fold_csv)); fold_rows.append(json.loads(fold_json.read_text(encoding="utf-8"))); imp_accum.append(np.load(fold_imp_npy))
            print(f"[RESUME] {run_name} fold {fk}", flush=True)
            continue

        set_seed(seed + fk)
        px_stats = compute_fold_pixel_stats(fold["train"], bags, cache) if is_paper else (0.485, 0.229)
        tr_ds = PaperBagDataset(fold["train"], cohort_df, pipe_tmp, bags, cache, task_mode, img_regime, train=True, pixel_stats=px_stats)
        va_ds = PaperBagDataset(fold["val"], cohort_df, pipe_tmp, bags, cache, task_mode, img_regime, train=False, pixel_stats=px_stats)
        tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_patients, generator=torch.Generator().manual_seed(seed + fk))
        va_loader = DataLoader(va_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_patients)

        model = BarthwalMultimodalNet(tab_dim=len(col_names_ref), d_model=64, n_steps=5).to(DEVICE)
        opt = torch.optim.AdamW([
            {"params": model.backbone.parameters(), "lr": lr_backbone, "weight_decay": 1e-4},
            {"params": [p for n, p in model.named_parameters() if not n.startswith("backbone.")], "lr": lr_tab_fusion, "weight_decay": 1e-4},
        ])
        scaler = torch.amp.GradScaler("cuda", enabled=(amp_fp16 and DEVICE.type == "cuda"))

        best_score, best_state, no_imp, best_ep = (-1.0, -float("inf")), None, 0, 0
        for ep in range(1, epochs + 1):
            t0 = time.perf_counter(); model.train()
            for items in tr_loader:
                opt.zero_grad(set_to_none=True)
                y_true = torch.tensor([it["y_bin"] for it in items], device=DEVICE, dtype=torch.float32)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(amp_fp16 and DEVICE.type == "cuda")):
                    logit, m_loss, _ = model(items, DEVICE)
                loss = F.binary_cross_entropy_with_logits(logit.float(), y_true) + lambda_sparse * m_loss
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

            model.eval(); va_probs, va_ys, va_losses = [], [], []
            with torch.inference_mode():
                for items in va_loader:
                    y_true = torch.tensor([it["y_bin"] for it in items], device=DEVICE, dtype=torch.float32)
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(amp_fp16 and DEVICE.type == "cuda")):
                        logit, _, _ = model(items, DEVICE)
                    va_probs.extend(torch.sigmoid(logit.float()).cpu().tolist()); va_ys.extend([int(it["y_bin"]) for it in items])
                    va_losses.append(float(F.binary_cross_entropy_with_logits(logit.float(), y_true).item()))
            cand = (float(roc_auc_score(va_ys, va_probs)), -float(np.mean(va_losses)))
            if cand > best_score:
                best_score, best_state, no_imp, best_ep = cand, copy.deepcopy(model.state_dict()), 0, ep
            else:
                no_imp += 1
            if ep in (1, epochs) or ep % 5 == 0 or no_imp >= patience:
                print(f"{run_name} | fold {fk} | ep {ep}/{epochs} | val_auc={cand[0]:.4f} | best={best_ep} | {time.perf_counter()-t0:.1f}s", flush=True)
            if no_imp >= patience:
                break

        model.load_state_dict(best_state); model.eval()
        rows, imp_vecs = [], []
        with torch.inference_mode():
            for items in va_loader:
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(amp_fp16 and DEVICE.type == "cuda")):
                    logit, _, m_explain = model(items, DEVICE)
                probs = torch.sigmoid(logit.float()).cpu().numpy()
                imp_vecs.append((m_explain / m_explain.sum(dim=-1, keepdim=True).clamp_min(1e-8)).cpu().numpy())
                for it, p in zip(items, probs):
                    rows.append({"patient_id": int(it["patient_id"]), "raw_label": int(it["raw_label"]), "label": int(it["y_bin"]), "prob": float(p), "p_0": 1.0 - float(p), "p_1": float(p), "pred": int(p >= 0.50), "n_images": len(it["image_ids"]), "fold": fk})
        f_df = pd.DataFrame(rows)
        fold_imp = np.concatenate(imp_vecs, axis=0).mean(axis=0)
        m = compute_binary_metrics(f_df["label"], f_df["prob"], f_df["pred"])
        m_row = {"fold": fk, "best_epoch": best_ep, **{k: v for k, v in m.items() if np.isscalar(v)}}
        out_dir.mkdir(parents=True, exist_ok=True); f_df.to_csv(fold_csv, index=False); save_json(fold_json, m_row); np.save(fold_imp_npy, fold_imp)
        oof_parts.append(f_df); fold_rows.append(m_row); imp_accum.append(fold_imp)
        del model, opt, scaler, tr_ds, va_ds; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    res = save_binary_run_artifacts(out_dir, run_name, pd.concat(oof_parts, ignore_index=True), fold_rows, {"paper": "Barthwal2026", "regime": regime, "task_mode": task_mode, "n_features": len(col_names_ref)})
    mean_imp = np.mean(np.stack(imp_accum, axis=0), axis=0)
    imp_df = pd.DataFrame({"feature": col_names_ref, "tabnet_importance": mean_imp}).sort_values("tabnet_importance", ascending=False).reset_index(drop=True)
    imp_df.to_csv(out_dir / "tabnet_feature_importance.csv", index=False)
    res["importance"] = imp_df
    return res

barthwal_results = {}
if RUN_BARTHWAL_PAPER or RUN_BARTHWAL_CLEAN:
    barthwal_cache_224 = build_barthwal_cache(manifest, ROOT / "US_Pictures_clean", PM3_DIR / "cache", size=224) if RUN_BARTHWAL_PAPER else {}
if RUN_BARTHWAL_PAPER:
    for tm in ("binary_diag", "binary_sev"):
        barthwal_results[f"barthwal_paper_leaky53_{tm}"] = run_barthwal_cv("paper_leaky53", tm, folds, cohort, bags_all, image_cache_clean, barthwal_cache_224, epochs=25, patience=7)
if RUN_BARTHWAL_CLEAN:
    for tm in ("binary_diag", "binary_sev"):
        barthwal_results[f"barthwal_our_clean28_{tm}"] = run_barthwal_cv("our_clean28", tm, folds, cohort, bags_all, image_cache_clean, image_cache_clean, epochs=25, patience=7)


## 4. Tabel Perbandingan Resmi (Target 1, Target 2, Target 3) + Statistik Berpasangan


In [ ]:
def load_pm3_binary_oof(run_folder: str, task_mode: str) -> Optional[pd.DataFrame]:
    p = RUNS_DIR / run_folder / "oof_predictions.csv"
    if not p.is_file():
        return None
    df = pd.read_csv(p).sort_values("patient_id").reset_index(drop=True)
    if "prob" in df.columns:
        return df
    out = pd.DataFrame({
        "patient_id": df["patient_id"].astype(int),
        "raw_label": df["raw_label"].astype(int),
        "label": (df["raw_label"].astype(int) > 0).astype(int) if task_mode == "binary_diag" else (df["raw_label"].astype(int) == 2).astype(int),
        "prob": (1.0 - df["p_0"].to_numpy(float)) if task_mode == "binary_diag" else df["p_2"].to_numpy(float),
        "fold": df["fold"].astype(int),
        "n_images": df["n_images"].astype(int),
    })
    out_cal, _ = calibrate_binary_oof(out)
    return out_cal

def load_rep(name: str) -> Optional[pd.DataFrame]:
    p = OUT_ROOT / name / "oof_predictions.csv"
    return pd.read_csv(p).sort_values("patient_id").reset_index(drop=True) if p.is_file() else None

def summarize_entries(entries: List[Tuple[str, str, Optional[pd.DataFrame]]], ref_name: str, task_mode: str) -> pd.DataFrame:
    loaded = [(grp, nm, df) for grp, nm, df in entries if df is not None]
    ref_df = next((df for _, nm, df in loaded if nm == ref_name), None)
    if ref_df is None and loaded:
        ref_df = loaded[0][2]
        ref_name = loaded[0][1]
    rows = []
    for grp, nm, df in loaded:
        y = df["label"].to_numpy(int)
        pr = df["prob"].to_numpy(float)
        pd_cal = df["pred_calibrated"].to_numpy(int) if "pred_calibrated" in df.columns else (pr >= 0.50).astype(int)
        m = compute_binary_metrics(y, pr, pd_cal)
        row = {
            "group": grp, "model": nm, "task": task_mode, "n": m["n"], "n_pos": m["n_pos"],
            "auroc": m["auroc"], "aupr": m["aupr"], "accuracy": m["accuracy"],
            "f1": m["f1"], "macro_f1": m["macro_f1"],
            "sensitivity": m["sensitivity"], "specificity": m["specificity"],
            "brier": m["brier"], "ece": m["ece"],
        }
        if ref_df is not None and nm != ref_name:
            d_auc, lo, hi, p_auc = paired_bootstrap_delta_auroc(y, ref_df["prob"].to_numpy(float), pr, n_boot=3000, seed=1)
            ref_pred = ref_df["pred_calibrated"].to_numpy(int) if "pred_calibrated" in ref_df.columns else (ref_df["prob"].to_numpy(float) >= 0.50).astype(int)
            row["delta_auroc_ref_minus_model"] = d_auc
            row["delta_ci95"] = f"[{lo:+.4f}, {hi:+.4f}]"
            row["p_boot_auroc"] = p_auc
            row["p_mcnemar"] = exact_mcnemar_p(y, ref_pred, pd_cal)
        rows.append(row)
    return pd.DataFrame(rows)

# --- TABEL 1: Image-Only PM3 (Joint Binary & 09 APF Binary) vs MVC-Mean & MVC-LSTM ---
t1_parts = []
for tm in ("binary_diag", "binary_sev"):
    entries_1 = [
        ("Ours (09 APF Bin)",   "PM3-09-APF Binary (Image-Only)",  load_rep(f"pm3_apf09_bin_image_only_{tm}")),
        ("Ours (Joint Bin)",    "PM3-Joint Binary (Image-Only)",   load_rep(f"pm3_joint_bin_image_only_{tm}")),
        ("Ours (B2 Ref)",       "PM3 B2 Image-Only CrossAttn",     load_pm3_binary_oof(f"02_b2_image_only_{tm}_cross_attn", tm)),
        ("Paper 1 Sequential",  "MVCBM-Seq-Mean (Paper 1)",        load_rep(f"mvcbm_seq_image_only_{tm}_avg")),
        ("Paper 1 Sequential",  "MVCBM-Seq-LSTM (Paper 1)",        load_rep(f"mvcbm_seq_image_only_{tm}_lstm")),
        ("Paper 1 Joint",       "MVCBM-Joint-Mean (Paper 1)",      load_rep(f"mvcbm_joint_image_only_{tm}_avg")),
        ("Paper 1 Joint",       "MVCBM-Joint-LSTM (Paper 1)",      load_rep(f"mvcbm_joint_image_only_{tm}_lstm")),
    ]
    t1_parts.append(summarize_entries(entries_1, "PM3-09-APF Binary (Image-Only)", tm))
table1_df = pd.concat(t1_parts, ignore_index=True)
table1_df.to_csv(OUT_ROOT / "table1_image_only_vs_mvc.csv", index=False)
print("=== TABEL 1: Image-Only PM3 (Joint & 09 APF Binary) vs MVC-Mean & MVC-LSTM (N=578) ===")
display(table1_df.round(4))

# --- TABEL 2: Multimodal PM3 (Joint Binary & 09 APF Binary) vs MVC-Mean & MVC-LSTM ---
t2_parts = []
for tm in ("binary_diag", "binary_sev"):
    entries_2 = [
        ("Ours (09 APF Bin)",   "PM3-09-APF Binary (Multimodal)",  load_rep(f"pm3_apf09_bin_multimodal_{tm}")),
        ("Ours (Joint Bin)",    "PM3-Joint Binary (Multimodal)",   load_rep(f"pm3_joint_bin_multimodal_{tm}")),
        ("Ours (09 Hier Proj)", "PM3-09-APF Hierarchical (proj)",  load_pm3_binary_oof("09_pm3_dualstage7_apf", tm)),
        ("Ours (Joint Hier)",   "PM3-Joint Hierarchical (proj)",   load_pm3_binary_oof("02_backbone_stage7_only", tm)),
        ("Paper 1 Img-Only",    "MVCBM-Seq-Mean ImgOnly",          load_rep(f"mvcbm_seq_image_only_{tm}_avg")),
        ("Paper 1 Img-Only",    "MVCBM-Seq-LSTM ImgOnly",          load_rep(f"mvcbm_seq_image_only_{tm}_lstm")),
        ("Paper 1 MM Seq",      "MVCBM-Seq-Mean Multimodal",       load_rep(f"mvcbm_seq_multimodal_{tm}_avg")),
        ("Paper 1 MM Seq",      "MVCBM-Seq-LSTM Multimodal",       load_rep(f"mvcbm_seq_multimodal_{tm}_lstm")),
        ("Paper 1 MM Joint",    "MVCBM-Joint-Mean Multimodal",     load_rep(f"mvcbm_joint_multimodal_{tm}_avg")),
        ("Paper 1 MM Joint",    "MVCBM-Joint-LSTM Multimodal",     load_rep(f"mvcbm_joint_multimodal_{tm}_lstm")),
    ]
    t2_parts.append(summarize_entries(entries_2, "PM3-09-APF Binary (Multimodal)", tm))
table2_df = pd.concat(t2_parts, ignore_index=True)
table2_df.to_csv(OUT_ROOT / "table2_multimodal_vs_mvc.csv", index=False)
print("=== TABEL 2: Multimodal PM3 (Joint & 09 APF Binary) vs MVC-Mean & MVC-LSTM (N=578) ===")
display(table2_df.round(4))

# --- TABEL 3: Bukti Overklaim — PM3 (Joint & 09 APF) vs Barthwal et al. (2026) (Paper vs Clean) ---
t3_parts = []
for tm in ("binary_diag", "binary_sev"):
    entries_3 = [
        ("Paper 2 (Overclaim)",  "Barthwal2026 (Paper Preproc: leaky_53 + 224)", load_rep(f"barthwal_paper_leaky53_{tm}")),
        ("Paper 2 (Fair Clean)", "Barthwal2026 (Our Clean: clean_28 + USConfig)", load_rep(f"barthwal_our_clean28_{tm}")),
        ("Ours (09 APF Bin)",    "PM3-09-APF Binary (Multimodal, clean_28)",      load_rep(f"pm3_apf09_bin_multimodal_{tm}")),
        ("Ours (Joint Bin)",     "PM3-Joint Binary (Multimodal, clean_28)",       load_rep(f"pm3_joint_bin_multimodal_{tm}")),
    ]
    t3_parts.append(summarize_entries(entries_3, "PM3-09-APF Binary (Multimodal, clean_28)", tm))
table3_df = pd.concat(t3_parts, ignore_index=True)
table3_df.to_csv(OUT_ROOT / "table3_overclaim_barthwal_vs_pm3.csv", index=False)
print("=== TABEL 3: Bukti Overklaim Barthwal et al. (2026) vs PM3 (N=578) ===")
display(table3_df.round(4))


## 5. Visualisasi Perbandingan Publikasi & Audit Fitur TabNet Overklaim


In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

def plot_grouped_auroc(df: pd.DataFrame, title: str, out_png: Path):
    if df is None or df.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
    for ax, tm, sub_title in zip(axes, ("binary_diag", "binary_sev"), ("Binary Diagnosis (App vs No-App)", "Binary Severity (Comp vs All, N=578)")):
        sub = df.loc[df["task"] == tm].copy()
        if sub.empty:
            continue
        y_pos = np.arange(len(sub))
        colors = ["#1f77b4" if "Ours" in g or "PM3" in m else ("#d62728" if "Overclaim" in g else "#7f7f7f")
                  for g, m in zip(sub["group"], sub["model"])]
        ax.barh(y_pos, sub["auroc"], color=colors, alpha=0.88, height=0.62)
        for yp, v in zip(y_pos, sub["auroc"]):
            ax.text(min(v + 0.008, 0.98), yp, f"{v:.4f}", va="center", fontsize=8.5)
        ax.set_yticks(y_pos); ax.set_yticklabels(sub["model"], fontsize=8.5)
        ax.invert_yaxis(); ax.set_xlim(0.50, 1.03); ax.set_xlabel("OOF AUROC")
        ax.set_title(sub_title); ax.grid(axis="x", linestyle="--", alpha=0.35)
    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout(); fig.savefig(out_png, dpi=300, bbox_inches="tight"); plt.close(fig)

plot_grouped_auroc(table1_df, "Perbandingan 1: Image-Only PM3 (Joint & 09 APF) vs MVC-Mean & MVC-LSTM", OUT_ROOT / "fig1_image_only_vs_mvc.png")
plot_grouped_auroc(table2_df, "Perbandingan 2: Multimodal PM3 (Joint & 09 APF) vs MVC-Mean & MVC-LSTM", OUT_ROOT / "fig2_multimodal_vs_mvc.png")

imp_p = OUT_ROOT / "barthwal_paper_leaky53_binary_diag" / "tabnet_feature_importance.csv"
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
if not table3_df.empty:
    sub = table3_df.loc[table3_df["task"] == "binary_diag"].copy()
    if not sub.empty:
        yp = np.arange(len(sub))
        cols = ["#d62728" if "leaky_53" in m else ("#ff7f0e" if "Clean:" in m else "#1f77b4") for m in sub["model"]]
        axes[0].barh(yp, sub["auroc"], color=cols, alpha=0.9, height=0.55)
        for yv, val in zip(yp, sub["auroc"]):
            axes[0].text(min(val + 0.008, 0.99), yv, f"{val:.4f}", va="center", fontsize=9)
        axes[0].set_yticks(yp); axes[0].set_yticklabels(sub["model"], fontsize=8.5)
        axes[0].invert_yaxis(); axes[0].set_xlim(0.55, 1.05); axes[0].set_xlabel("OOF AUROC (Binary Diagnosis)")
        axes[0].set_title("Bukti Overklaim: Drop Saat Kebocoran Fitur Dibersihkan")
        axes[0].grid(axis="x", linestyle="--", alpha=0.35)

LEAKY_MARKERS = {
    "Length_of_Stay", "Alvarado_Score", "Paedriatic_Appendicitis_Score",
    "Appendix_on_US", "Appendix_Diameter", "Free_Fluids", "Appendix_Wall_Layers",
    "Target_Sign", "Appendicolith", "Perfusion", "Perforation",
    "Surrounding_Tissue_Reaction", "Appendicular_Abscess", "Abscess_Location",
    "Pathological_Lymph_Nodes", "Lymph_Nodes_Location", "Bowel_Wall_Thickening",
    "Conglomerate_of_Bowel_Loops", "Ileus", "Coprostasis", "Meteorism",
    "Enteritis", "Gynecological_Findings", "US_Performed",
}
if imp_p.is_file():
    imp_df = pd.read_csv(imp_p).head(15)
    yp2 = np.arange(len(imp_df))
    c2 = ["#d62728" if f in LEAKY_MARKERS else "#4c72b0" for f in imp_df["feature"]]
    axes[1].barh(yp2, imp_df["tabnet_importance"], color=c2, alpha=0.9, height=0.6)
    axes[1].set_yticks(yp2); axes[1].set_yticklabels(imp_df["feature"], fontsize=8.5)
    axes[1].invert_yaxis(); axes[1].set_xlabel("Mean Normalized TabNet Sparse Mask")
    axes[1].set_title("Top-15 Fitur TabNet Rezim Paper (Merah = Fitur Bocor)")
    axes[1].grid(axis="x", linestyle="--", alpha=0.35)
else:
    axes[1].text(0.5, 0.5, "Jalankan RUN_BARTHWAL_PAPER untuk melihat TabNet Importance", ha="center", va="center")
    axes[1].axis("off")

fig.tight_layout(); fig.savefig(OUT_ROOT / "fig3_overclaim_proof_and_tabnet_importance.png", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Visualisasi disimpan ke:", OUT_ROOT, flush=True)
